# T12 Parsing Workflow — Prototype

Building the new parsing workflow one **module per cell**, so each stage can be run and inspected on its own.

This notebook runs on **one file at a time**: pick `INPUT_FILE` in the setup cell, then run the cells top-to-bottom. Every module works on that single loaded file. To try a different file, change `INPUT_FILE` and re-run all.

Reference architecture: `app/t12_pipeline/docs/proposals/t12_parsing_proposal.md`

Stages built so far:
1. **Setup** — imports, paths, pick the input file
2. **Module 1 — File load** — open the workbook, expose metadata for pre-flight
3. **Module 2 — Pre-flight** — is this file a rent roll / Model T12? (reject) or a real T12? (proceed)
4. **Module 3 — Layer 1: Raw extraction** — read Excel once, losslessly, into `cell_grid` (single source of truth)
5. **Module 4 — Format detection** — score `cell_grid` against known layouts; low score → unknown
6. **Modules 5–7 — Layer 2: Signal extraction (known formats)** — per-format feature vectors from `cell_grid` (`code_in_a_desc_in_b`, `embedded_code_in_a`, `no_code`)
7. **Module 8 — Unknown route** — **(8a)** deterministic `derive_layout` candidate; **(8b)** a full structure-map agent (`gpt-4.1`) verifies & corrects it against the raw grid and adds the section rows; **(8c)** reconcile (counting → deterministic, interpretive → AI) + **arithmetic-validated subsection hierarchy** + arithmetic gate
8. **Module 9 — Dynamic signal extraction (unknown route)** — drive the extractor off the final structure map; emit the **same feature vector** Modules 5–7 produce (`column_depth` carries TREE hierarchy)
9. **Module 10 — Row classifier** — score-based 7-class (METADATA / SECTION_HEADER / SUBSECTION_HEADER / DATA / SUBSECTION_SUBTOTAL / SECTION_SUBTOTAL / COMPUTED_METRIC); prints all 7 scores; **leaf-DATA reconciliation** self-check
10. **Module 11 — Section assignment** — **two independent branches** find the section boundaries (deterministic keyword match ⟂ AI structure agent); agree → HIGH confidence, disagree → review. Then a state machine tags each row revenue / expense / capex
11. **Module 12 — NOI verification** — bottom-up (Σ DATA by section) vs top-down (vendor totals), compared **per month and total** (mirrors src `compute_noi` + `compare_noi`); the arithmetic tie-breaker in the triangulation
12. **Module 13 — AI Failure Diagnosis** — on NOI FAIL, a `gpt-4.1` agent gets the full grid + our labels + the discrepancy and returns DATA_PROBLEM vs CLASSIFICATION_PROBLEM (with the culprit rows)


In [20]:
import re
import io
import contextlib
from pathlib import Path

import openpyxl

SAMPLE_DIR = Path(
    "/Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/"
    "master_docs/client_sent_docs/6th_june_new_t12_docs/T12"
)

# ── Pick ONE file to run the whole workflow on ────────────────────────────────
# Change the active INPUT_FILE line (or uncomment one below) and re-run all cells.
#
# ============================ VALIDATED unknown-format test set ================
# Hierarchical / unknown layout → the full AI branch (Modules 8a…14). These
# reconcile 7/7: six pass clean at Module 12; Eleven11 is the one that FAILS the
# NOI check and is then FIXED by the feedback loop (Modules 13-14).
# INPUT_FILE = SAMPLE_DIR / "New/Cardinal Seller T12.xlsx"                                 # clean pass
# INPUT_FILE = SAMPLE_DIR / "New/Holden Seller T12.xlsx"                                  # clean pass (bold-only subtotals)
# INPUT_FILE = SAMPLE_DIR / "New/Arbor Gates at Buckhead Seller T12.xlsx"                 # clean pass
# INPUT_FILE = SAMPLE_DIR / "New/Mesquite Seller T-12.xlsx"                               # clean pass
# INPUT_FILE = SAMPLE_DIR / "New/Crystal Lake_T-12_04.2026.xlsx"                          # clean pass
# INPUT_FILE = SAMPLE_DIR / "Previously Provided/Chateau in the  Woods T12 08-25.xlsx"    # clean pass
INPUT_FILE = SAMPLE_DIR / "New/Eleven11 Lexington at Flower Mound Seller T12.xlsx"      # ★ FAILS Module 12 → FIXED by the feedback loop
#
# ============================ Known-format examples ===========================
# Routed by Module 4 to a per-format extractor (NOT the AI branch):
# INPUT_FILE = SAMPLE_DIR / "New/Lincoln Townhomes Seller T12.xlsx"                       # code_in_a_desc_in_b
# INPUT_FILE = SAMPLE_DIR / "New/Creekside Seller T12.xlsx"                               # embedded_code_in_a
# INPUT_FILE = SAMPLE_DIR / "New/Ladera Seller T12.xlsx"                                  # no_code
#
# ===================== More Seller T12s to try (untested) =====================
# Candidates for the unknown branch — not yet validated; use to stress-test.
# INPUT_FILE = SAMPLE_DIR / "New/Lockhart Seller T-12.xlsx"
# INPUT_FILE = SAMPLE_DIR / "New/Mirador Seller T12 .xlsx"
# INPUT_FILE = SAMPLE_DIR / "New/RESERVE AT WHITE ROCK Seller T12.xlsx"
# INPUT_FILE = SAMPLE_DIR / "New/TRAILS AT WHITE ROCK Seller T12.xlsx"
# INPUT_FILE = SAMPLE_DIR / "New/River Point Seller T12 Feb.xlsx"
# INPUT_FILE = SAMPLE_DIR / "New/River Point West Seller T12.xlsx"
# INPUT_FILE = SAMPLE_DIR / "New/Wythe Seller T12.xlsx"
# INPUT_FILE = SAMPLE_DIR / "Previously Provided/Brownsburg Crossing T12 05-25.xlsx"
# INPUT_FILE = SAMPLE_DIR / "Previously Provided/Carmel Woods T12 09-25.xlsx"
# INPUT_FILE = SAMPLE_DIR / "Previously Provided/Echo Park at Perry Crossing T12 12-24.xlsx"
# INPUT_FILE = SAMPLE_DIR / "Previously Provided/Pebble Brook Village T12 09-25.xlsx"
# INPUT_FILE = SAMPLE_DIR / "Previously Provided/Pointe Inverness T12 11-25.xlsx"
# INPUT_FILE = SAMPLE_DIR / "Previously Provided/Willows of Coventry T12 11-25.xlsx"
#
# ===================== Should be REJECTED by pre-flight ========================
# INPUT_FILE = SAMPLE_DIR / "Previously Provided/Jasmine - RR - 3.4.26.xlsx"              # rent roll (not a T12)
# INPUT_FILE = SAMPLE_DIR / "New/Cardinal Model T12.xlsx"                                 # model T12 (not a seller T12)
# INPUT_FILE = SAMPLE_DIR / "New/02242026_Grande Reserve at Geist_V3- VM..xlsx"           # parsed-output artifact

print(f"Input file : {INPUT_FILE.name}")
print(f"Exists     : {INPUT_FILE.exists()}")


# ── output plumbing: file contract between stages (mirrors the src pipeline) ───
import json
OUTPUT_DIR = Path.cwd() / "workflow_output"

def save_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)

def load_json(path):
    with open(path) as f:
        return json.load(f)

def load_cell_grid(path):
    """Load cell_grid and restore integer column keys (JSON stringifies dict keys)."""
    grid = load_json(path)
    for row in grid["rows"]:
        row["cells"] = {int(k): v for k, v in row["cells"].items()}
    return grid


Input file : Eleven11 Lexington at Flower Mound Seller T12.xlsx
Exists     : True


## Module 1 — File load

Open a workbook and hand back the active sheet plus a little metadata (sheet names, size). Everything downstream works off `ws`.

We load with `data_only=True` so cells return their computed **values** (not formulas) — that's what the value-based signals need.

In [21]:
def load_file(path):
    """Open an .xlsx and return (workbook, active_sheet, info).
    info: {file_name, sheet_names, active_sheet, max_row, max_col}
    Loaded data_only=True so numeric cells return computed values."""
    path = Path(path)
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb.active
    info = {
        "file_name":    path.name,
        "sheet_names":  wb.sheetnames,
        "active_sheet": ws.title,
        "max_row":      ws.max_row,
        "max_col":      ws.max_column,
    }
    return wb, ws, info

## Module 2 — Pre-flight check

Before any parsing, decide whether the loaded file is even a T12 input. Rejected if it's a **rent roll** (filename or a sheet name matches `RR` / `Rent Roll`) or a **Model T12** file (filename ends in `Model T12.xlsx` — a canonical target, not an input).

Runs on the file loaded above and prints the verdict. If it's rejected here, the modules below wouldn't run in the real pipeline.

In [22]:
RENT_ROLL_RE = re.compile(r"(?:\bRR\b|_RR_|Rent[\s_]?Roll)", re.IGNORECASE)
MODEL_T12_RE = re.compile(r"Model[\s_]?T12\.xlsx$", re.IGNORECASE)


def preflight(path, sheet_names):
    """Decide whether a file should proceed into the T12 pipeline.
    Returns (verdict, kind) where verdict is 'proceed' or 'reject',
       kind ∈ {t12_candidate, rent_roll, model_t12}."""
    name = Path(path).name

    if MODEL_T12_RE.search(name):
        return "reject", "model_t12"          # canonical target, not an input
    if RENT_ROLL_RE.search(name):
        return "reject", "rent_roll"          # rent roll by filename
    if any(RENT_ROLL_RE.search(s) for s in sheet_names):
        return "reject", "rent_roll"          # rent roll by sheet name

    return "proceed", "t12_candidate"

## Module 3 — Layer 1: Raw extraction → `cell_grid`

Read the Excel **once**, losslessly, into `cell_grid` — the single source of truth every later module reads. Nothing downstream reopens the workbook (mirrors the `src` pipeline: read the xlsx once, everything else works from the structure).

For each cell in the **used range** we keep everything it holds: `value`, `formula`, `data_type`, `number_format`, font (`bold` / `italic` / `underline` / `size` / `color`), alignment (`align_h` / `indent`), `fill_color`, `border_top` / `border_bottom` (the accountant's "sum line"), `comment`, `hyperlink`. Plus worksheet-level facts: `merged_ranges` (critical for TREE layouts) and `hidden_rows` / `hidden_cols`.

- **Used range**, not Excel's million-row max — capped at 5000 rows / 50 cols to guard against a phantom `max_row`.
- Reads the file twice internally (values via `data_only=True`, formulas via `data_only=False`) — still Excel touched in exactly one stage.
- The original `.xlsx` is retained as the ultimate lossless source; `cell_grid` is the structured working copy.

Downstream reads `cell_grid` through the helpers `grid_row_map` / `gcell` / `gvalue`. Runs only if pre-flight passed.

In [23]:
from openpyxl.utils import get_column_letter

MAX_SCAN_ROWS, MAX_SCAN_COLS = 5000, 50   # guard against a phantom max_row / max_col


def _color(cobj):
    """Safe RGB string from an openpyxl color object, or None."""
    if cobj is None:
        return None
    rgb = getattr(cobj, "rgb", None)
    return rgb if isinstance(rgb, str) else None


def extract_cell_grid(path):
    """Layer 1 — read the workbook ONCE, losslessly, into cell_grid.
    Reads values (data_only=True) and formulas (data_only=False)."""
    wb_v = openpyxl.load_workbook(path, data_only=True);  ws_v = wb_v.active
    wb_f = openpyxl.load_workbook(path, data_only=False); ws_f = wb_f.active
    max_row = min(ws_v.max_row, MAX_SCAN_ROWS)
    max_col = min(ws_v.max_column, MAX_SCAN_COLS)

    rows = []
    for r in range(1, max_row + 1):
        cells = {}
        for c in range(1, max_col + 1):
            cv = ws_v.cell(row=r, column=c)
            fv = ws_f.cell(row=r, column=c).value
            formula = fv if (isinstance(fv, str) and fv.startswith("=")) else None
            bt = cv.border.top.style    if (cv.border and cv.border.top)    else None
            bb = cv.border.bottom.style if (cv.border and cv.border.bottom) else None
            fill_color = _color(cv.fill.fgColor) if (cv.fill and cv.fill.patternType) else None
            # keep any cell with content or non-default formatting; drop truly-empty cells
            if not ((cv.value is not None) or formula or bt or bb or fill_color or cv.comment or cv.hyperlink):
                continue
            cells[c] = {
                "col": c, "letter": get_column_letter(c),
                "value": cv.value, "formula": formula,
                "data_type": cv.data_type, "number_format": cv.number_format,
                "bold": bool(cv.font.bold) if cv.font else False,
                "italic": bool(cv.font.italic) if cv.font else False,
                "underline": cv.font.underline if cv.font else None,
                "font_size": cv.font.size if cv.font else None,
                "font_color": _color(cv.font.color) if cv.font else None,
                "align_h": cv.alignment.horizontal if cv.alignment else None,
                "indent": (cv.alignment.indent if cv.alignment else 0) or 0,
                "fill_color": fill_color,
                "border_top": bt, "border_bottom": bb,
                "comment": cv.comment.text if cv.comment else None,
                "hyperlink": cv.hyperlink.target if cv.hyperlink else None,
            }
        if cells:
            rows.append({"row": r, "cells": cells})

    grid = {
        "sheet": ws_v.title, "max_row": max_row, "max_col": max_col,
        "merged_ranges": [str(m) for m in ws_v.merged_cells.ranges],
        "hidden_rows": [r for r, d in ws_v.row_dimensions.items() if d.hidden],
        "hidden_cols": [k for k, d in ws_v.column_dimensions.items() if d.hidden],
        "rows": rows,
    }
    wb_v.close(); wb_f.close()
    return grid


# ── helpers every downstream module uses to read the grid ─────────────────────
def grid_row_map(grid):
    """{sheet_row_number: row_record} for O(1) lookup by row number."""
    return {row["row"]: row for row in grid["rows"]}


def gcell(grow, col):
    """Cell dict at 1-based column `col` in a row record, or None."""
    return grow["cells"].get(col) if grow else None


def gvalue(grow, col):
    """Value at 1-based column `col`, or None."""
    c = gcell(grow, col)
    return c["value"] if c else None

## Module 4 — Format detection (signal-based, per column)

Reads `cell_grid`. For each of the first ~60 rows and each column A–F, categorize that cell into one kind, then normalize to a fraction — giving a `(kind × column)` grid.

| Kind | Cell looks like |
|---|---|
| `pure_code` | *only* an account code, e.g. `4350-0000` |
| `embedded_code` | account code **and** description together, e.g. `5120 - Gross Rent` |
| `foreign_code` | a code style we don't support — dot `5120.0000`, colon `4010:`, MRI `1-5000-0005`, 5-digit `41000-000` |
| `indented_text` | indented plain text with no code |
| `plain_text` | flush-left plain text (headers / metadata) |
| `numeric` | a number — **informational**: shows where the value columns start. Not used in the decision. |

A cell goes into exactly one kind (checked in priority order), so nothing double-counts.

**Normalization.** `rows_with_text` = how many of the scanned rows had at least one **text** cell. Every fraction is `count / rows_with_text` (the demo prints both as `count=fraction`).

**How it decides.** Known formats are all anchored in **column A**, so their scores read straight off the column-A signals:
- `code_in_a_desc_in_b` ← `pure_code` in column A
- `embedded_code_in_a` ← `embedded_code` in column A
- `no_code` ← `indented_text` in column A
- `unknown` ← the largest `foreign_code` in any column

Then two rules give `unknown` on its own merits (no penalties):
1. **Structural gate** — if codes *dominate a column other than A* (hierarchical / TREE) → `unknown`.
2. **Threshold / dominance** — if the best known score is below `THRESHOLD`, or `unknown` outscores it → `unknown`.

In [24]:
# ── 4a. Per-cell kind test + per-(kind, column) signal extraction ─────────────
# Reads cell_grid (Layer 1) — no Excel access here.

PURE_CODE      = re.compile(r"^\d{4}-\d{2,4}$")             # 4350-0000  (whole cell)
EMBEDDED_CODE  = re.compile(r"^\s*\d{4}\s*-\s*\d*\s*-?\s*")  # 5120 - Desc (code prefix)
EMBEDDED_STRIP = r"^\s*\d{4}\s*-\s*\d*\s*-?\s*"             # strip prefix to test for text
FOREIGN_CODES = {
    "dot":    re.compile(r"^\s*\d{4}\.\d"),      # 5120.0000     Cardinal
    "colon":  re.compile(r"^\s*\d{4}:"),         # 4010:Desc     Holden
    "mri":    re.compile(r"^\s*\d-\d{4}-\d"),     # 1-5000-0005   Crystal Lake
    "5digit": re.compile(r"^\d{5}-\d"),          # 41000-000     Eleven11 / Mesquite
}

# Six kinds a cell can be. The five TEXT kinds drive the format decision;
# `numeric` is informational — it shows where the value columns are.
KINDS       = ["pure_code", "embedded_code", "foreign_code", "indented_text", "plain_text", "numeric"]
TEXT_KINDS  = {"pure_code", "embedded_code", "foreign_code", "indented_text", "plain_text"}
NCOLS       = 6                                   # scan columns A..F
COL_LETTERS = ["a", "b", "c", "d", "e", "f"]


def cell_kind(v):
    """Categorize ONE cell value into exactly one kind (priority order), or None."""
    if isinstance(v, (int, float)):
        return "numeric"
    if not (isinstance(v, str) and v.strip()):
        return None
    raw, s = v, v.strip()
    if PURE_CODE.match(s):
        return "pure_code"
    if any(p.match(raw) for p in FOREIGN_CODES.values()):
        return "foreign_code"
    if EMBEDDED_CODE.match(raw) and re.search(r"[A-Za-z]", re.sub(EMBEDDED_STRIP, "", raw)):
        return "embedded_code"
    if raw.startswith("  ") and re.search(r"[A-Za-z]", s) and not s[0].isdigit():
        return "indented_text"
    if re.search(r"[A-Za-z]", s):
        return "plain_text"
    return None


def extract_format_signals(grid, scan_rows=60):
    """For every column A-F over the first `scan_rows`, count how many text rows
    have a cell of each kind, then normalize. Reads cell_grid.
    Returns (counts, fractions, rows_with_text)."""
    rowmap = grid_row_map(grid)
    counts = {k: [0] * NCOLS for k in KINDS}
    rows_with_text = 0
    for r in range(1, min(grid["max_row"], scan_rows) + 1):
        grow = rowmap.get(r)
        had_text = False
        for c in range(NCOLS):
            k = cell_kind(gvalue(grow, c + 1))
            if k:
                counts[k][c] += 1
                if k in TEXT_KINDS:
                    had_text = True
        if had_text:
            rows_with_text += 1

    denom = max(rows_with_text, 1)
    fractions = {k: [round(counts[k][c] / denom, 3) for c in range(NCOLS)] for k in KINDS}
    return counts, fractions, rows_with_text


# ── 4b. Score & decide (per-column, no penalties) ─────────────────────────────

DECISION_THRESHOLD = 0.15   # best known-format score must clear this, else 'unknown'


def detect_format(grid, scan_rows=60):
    """Return {format, confidence, reason, scores, counts, fractions, rows_with_text}.
    Only the TEXT code-kinds in column A drive the decision; `numeric` is not used."""
    counts, F, rows_with_text = extract_format_signals(grid, scan_rows)

    # Known formats are anchored in column A → read their column-A fraction.
    scores = {
        "code_in_a_desc_in_b": F["pure_code"][0],
        "embedded_code_in_a":  F["embedded_code"][0],
        "no_code":             F["indented_text"][0],
        "unknown":             max(F["foreign_code"]),      # foreign style, any column
    }

    # Structural gate: where do codes dominate? If not column A, the file spreads
    # codes across columns (hierarchical / TREE) — not one of our flat formats.
    code_per_col = [F["pure_code"][c] + F["embedded_code"][c] + F["foreign_code"][c]
                    for c in range(NCOLS)]
    if sum(code_per_col) > 0:
        dom = max(range(NCOLS), key=lambda c: code_per_col[c])
        if dom != 0:
            return {"format": "unknown", "confidence": round(scores["unknown"], 3),
                    "reason": f"codes dominate column {COL_LETTERS[dom].upper()}, not A (hierarchical)",
                    "scores": scores, "counts": counts, "fractions": F, "rows_with_text": rows_with_text}

    best = max(scores, key=scores.get)
    if best == "unknown" or scores[best] < DECISION_THRESHOLD:
        return {"format": "unknown", "confidence": round(scores["unknown"], 3),
                "reason": "no known format cleared the threshold / foreign codes dominate",
                "scores": scores, "counts": counts, "fractions": F, "rows_with_text": rows_with_text}

    return {"format": best, "confidence": scores[best],
            "reason": f"{best} strongest in column A",
            "scores": scores, "counts": counts, "fractions": F, "rows_with_text": rows_with_text}

## Module 5 — Layer 2: Signal extraction (`code_in_a_desc_in_b`)

For a file detected as `code_in_a_desc_in_b`, read `cell_grid` and turn every row into a **feature vector** the classifier will use to decide data / subtotal / header. Structure for this format: **code in column A, description in column B, values from column C onward** (value columns auto-detected).

The signals extracted per row (chosen from the earlier 1,340-row analysis):

| Signal | What it is |
|---|---|
| `has_values` | whether the row has any number in the value columns |
| `value_count` | how many value columns hold a number |
| `is_bold` | whether the description cell is bold |
| `total_keyword` | whether the description contains "Total …" or a grand-total keyword (NOI, EGI, …) |
| `has_formula` | whether any value cell contains a formula (`=…`) |
| `whitespace_depth` | number of leading spaces before the description (its indentation) |
| `column_depth` | which column the description starts in — `0` here (always column B); varies only in TREE layouts |
| `all_caps` | whether the description is entirely uppercase |
| `position_in_file` | how far down the sheet the row sits (`0.0` = top, `1.0` = bottom) |
| `arithmetic_sum_of_above` | whether the row's value equals the sum of the data rows directly above it |

**Two passes:** pass 1 computes every signal except the last; pass 2 (`_apply_arithmetic`, shared by all extractors) computes `arithmetic_sum_of_above`, which needs a provisional idea of which rows are data (here: has values and no "Total" keyword).

Everything is read from `cell_grid` — the value, bold, and formula all live there, so no Excel is reopened. This module also defines the shared helpers `has_total_keyword`, `detect_value_columns`, `print_signal_rows`, `_apply_arithmetic` reused by Modules 6 and 7. Runs only when the detected format is `code_in_a_desc_in_b`.

In [25]:
# ── code_in_a_desc_in_b structure (known for this format) ─────────────────────
# Reads cell_grid (Layer 1). code in column A, description in column B, values from C.
CODE_COL, DESC_COL = 1, 2

TOTAL_KEYWORDS = [
    "net operating income", "effective gross income", "net rental revenue", "net rental income",
    "net income", "net rent revenue", "net cash flow", "collectible rental revenue",
    "adjusted gross possible rental revenue", "total net rent", "total income", "total revenue",
    "total expenses", "total operating expenses", "total operating revenues", "taxable income",
    "income after debt service",
]


def has_total_keyword(desc):
    d = desc.strip().lower()
    if d.startswith("total ") or d == "total":
        return True
    return any(re.search(r"\b" + re.escape(kw) + r"\b", d) for kw in TOTAL_KEYWORDS)


def total_keyword_hit(desc):
    """The specific total-keyword that fired — for evidence/logging — or None."""
    d = desc.strip().lower()
    if d.startswith("total ") or d == "total":
        return "total*"
    return next((kw for kw in TOTAL_KEYWORDS if re.search(r"\b" + re.escape(kw) + r"\b", d)), None)


def detect_value_columns(grid, start_col, min_hits=5):
    """Columns at/after start_col that hold numbers in enough rows = the value region.
    Reads cell_grid."""
    numeric = {}
    for grow in grid["rows"]:
        for c, cell in grow["cells"].items():
            if c >= start_col and isinstance(cell["value"], (int, float)):
                numeric[c] = numeric.get(c, 0) + 1
    return sorted(c for c, n in numeric.items() if n >= min_hits)


# Column order for the signal table: most-important first so it's easy to scan.
# arithmetic_sum_of_above leads (the key roll-up check), then the decision-driving
# signals, then depth/position. Any signal not listed here is appended at the end.
_SIGNAL_COL_ORDER = [
    "arithmetic_sum_of_above", "has_values", "is_bold", "total_keyword",
    "is_metric", "top_border", "has_formula", "all_caps", "in_metadata_band",
    "column_depth", "whitespace_depth", "is_subsection_subtotal",
    "is_subsection_header", "value_count", "position_in_file",
]


def print_signal_rows(rows, limit=None):
    """Print EVERY extracted signal per row — nothing hidden. Columns are ordered
    most-important-first (arithmetic_sum_of_above leads, then the decision-driving
    signals); internal helper keys (prefixed '_') and nested evidence (list/dict)
    are omitted; `row` and `description` come first."""
    if not rows:
        print("(no rows)")
        return
    present = [k for k in rows[0] if not k.startswith("_") and k not in ("row", "description")
               and not isinstance(rows[0][k], (list, dict))]
    keys = [k for k in _SIGNAL_COL_ORDER if k in present] + \
           [k for k in present if k not in _SIGNAL_COL_ORDER]
    header = f"{'row':>4}  {'description':<30}  " + "  ".join(keys)
    print(header)
    print("-" * len(header))
    for r in (rows if limit is None else rows[:limit]):
        cells = []
        for k in keys:
            v = r[k]
            if isinstance(v, bool):
                v = "Y" if v else "."
            cells.append(f"{str(v):>{len(k)}}")
        print(f"{r['row']:>4}  {r['description'][:30]:<30}  " + "  ".join(cells))


def _apply_arithmetic(rows):
    """Pass 2 — arithmetic_sum_of_above. Provisional data = has values and no
    'Total' keyword; a row scores True if its value equals the sum of the
    consecutive provisional-data rows directly above it. Format-agnostic.
    Also records the full evidence on row['arithmetic'] (which rows were summed,
    their values, the sum, the target, the diff and tolerance) so a Y/N can always
    be explained — e.g. 'row 21 = Σ rows 12..20 = 100,800 vs target 100,800 ✓'."""
    for row in rows:
        row["_prov_data"] = row["has_values"] and not row["total_keyword"]
    for i, row in enumerate(rows):
        target, match = row["_last_value"], False
        member_rows, member_vals, s, j = [], [], 0.0, i - 1
        while j >= 0 and rows[j]["_prov_data"]:
            v = rows[j]["_last_value"]
            if v is not None:
                s += v; member_rows.append(rows[j]["row"]); member_vals.append(round(v, 2))
            j -= 1
        tol = max(1.0, abs(target) * 0.005) if isinstance(target, (int, float)) else None
        if isinstance(target, (int, float)) and member_rows:
            match = abs(s - target) <= tol
        row["arithmetic_sum_of_above"] = match
        row["arithmetic"] = {
            "fired":         match,
            "member_rows":   member_rows[::-1],   # top-to-bottom order
            "member_values": member_vals[::-1],
            "sum":           round(s, 2) if member_rows else None,
            "target":        round(target, 2) if isinstance(target, (int, float)) else None,
            "diff":          round(abs(s - target), 2) if (isinstance(target, (int, float)) and member_rows) else None,
            "tolerance":     round(tol, 2) if tol is not None else None,
        }


def extract_row_signals(grid, value_cols):
    """Per-row feature vector for a code_in_a_desc_in_b sheet, reading cell_grid.
    Pass 1: self-contained + value signals. Pass 2: arithmetic_sum_of_above."""
    max_row = grid["max_row"]
    rows = []

    # ── pass 1 ──
    for grow in grid["rows"]:
        r = grow["row"]
        a = gvalue(grow, CODE_COL)
        b = gvalue(grow, DESC_COL)
        b_raw = str(b) if b is not None else ""
        desc  = b_raw.strip() if b_raw.strip() else (str(a).strip() if a else "")

        numeric = [gvalue(grow, c) for c in value_cols]
        numeric = [v for v in numeric if isinstance(v, (int, float))]
        if not desc and not numeric:
            continue

        desc_cell = gcell(grow, DESC_COL) if b_raw.strip() else gcell(grow, CODE_COL)
        is_bold = bool(desc_cell["bold"]) if desc_cell else False
        has_formula = any((gcell(grow, c) or {}).get("formula") for c in value_cols)
        letters = [ch for ch in desc if ch.isalpha()]

        rows.append({
            "row":              r,
            "description":      desc,
            "has_values":       len(numeric) > 0,
            "value_count":      len(numeric),
            "is_bold":          is_bold,
            "total_keyword":    has_total_keyword(desc),
            "has_formula":      bool(has_formula),
            "whitespace_depth": len(b_raw) - len(b_raw.lstrip()),
            "column_depth":     0,                       # desc always column B here
            "all_caps":         bool(letters) and all(ch.isupper() for ch in letters),
            "position_in_file": round(r / max_row, 2),
            "_last_value":      numeric[-1] if numeric else None,
        })

    _apply_arithmetic(rows)
    return rows

## Module 6 — Layer 2: Signal extraction (`embedded_code_in_a`)

Same feature schema as Module 5, reading `cell_grid`. Structure differs: **everything is in column A** (leading whitespace + code + description merged, e.g. `    5120 - Gross Possible Rent`), and **values start in column B**.

Per-row differences from `code_in_a_desc_in_b`:
- **description** = column A, with the embedded code prefix stripped (`5120 - Gross Rent` → `Gross Rent`); header/subtotal rows have no code, so nothing is stripped.
- **whitespace_depth** = leading spaces of column A (this is where the hierarchy lives).
- **value columns** start at B.
- `is_bold` comes from column A.

Reuses the shared helpers from Module 5 (`has_total_keyword`, `detect_value_columns`, `print_signal_rows`, `_apply_arithmetic`) and `EMBEDDED_STRIP` from Module 4. Runs only when the detected format is `embedded_code_in_a`.

In [26]:
# ── embedded_code_in_a structure ──────────────────────────────────────────────
# Reads cell_grid. Everything in column A (whitespace + code + description); values from B.
EMB_DESC_COL = 1


def extract_row_signals_embedded(grid, value_cols):
    """Per-row feature vector for an embedded_code_in_a sheet, reading cell_grid.
    Description = column A with the embedded code prefix stripped."""
    max_row = grid["max_row"]
    rows = []

    # ── pass 1 ──
    for grow in grid["rows"]:
        r = grow["row"]
        a = gvalue(grow, EMB_DESC_COL)
        a_raw = str(a) if a is not None else ""
        desc = re.sub(EMBEDDED_STRIP, "", a_raw).strip()   # strip whitespace + code prefix

        numeric = [gvalue(grow, c) for c in value_cols]
        numeric = [v for v in numeric if isinstance(v, (int, float))]
        if not desc and not numeric:
            continue

        desc_cell = gcell(grow, EMB_DESC_COL)
        is_bold = bool(desc_cell["bold"]) if desc_cell else False
        has_formula = any((gcell(grow, c) or {}).get("formula") for c in value_cols)
        letters = [ch for ch in desc if ch.isalpha()]

        rows.append({
            "row":              r,
            "description":      desc,
            "has_values":       len(numeric) > 0,
            "value_count":      len(numeric),
            "is_bold":          is_bold,
            "total_keyword":    has_total_keyword(desc),
            "has_formula":      bool(has_formula),
            "whitespace_depth": len(a_raw) - len(a_raw.lstrip()),
            "column_depth":     0,                       # everything in column A
            "all_caps":         bool(letters) and all(ch.isupper() for ch in letters),
            "position_in_file": round(r / max_row, 2),
            "_last_value":      numeric[-1] if numeric else None,
        })

    _apply_arithmetic(rows)
    return rows

## Module 7 — Layer 2: Signal extraction (`no_code`)

Same feature schema again, reading `cell_grid`. Structure: **description in column A** (indented plain text, no codes), **values start in column B**. Simplest of the three — nothing to strip.

Per-row differences:
- **description** = column A, stripped of leading whitespace (no code to remove).
- **whitespace_depth** = leading spaces of column A (carries the hierarchy).
- **value columns** start at B.
- `is_bold` from column A.

Reuses the shared helpers from Module 5. Runs only when the detected format is `no_code`.

In [27]:
# ── no_code structure ─────────────────────────────────────────────────────────
# Reads cell_grid. Description in column A (indented, no code); values from column B.
NOCODE_DESC_COL = 1


def extract_row_signals_no_code(grid, value_cols):
    """Per-row feature vector for a no_code sheet, reading cell_grid.
    Description = column A stripped of leading whitespace (no code to remove)."""
    max_row = grid["max_row"]
    rows = []

    # ── pass 1 ──
    for grow in grid["rows"]:
        r = grow["row"]
        a = gvalue(grow, NOCODE_DESC_COL)
        a_raw = str(a) if a is not None else ""
        desc = a_raw.strip()

        numeric = [gvalue(grow, c) for c in value_cols]
        numeric = [v for v in numeric if isinstance(v, (int, float))]
        if not desc and not numeric:
            continue

        desc_cell = gcell(grow, NOCODE_DESC_COL)
        is_bold = bool(desc_cell["bold"]) if desc_cell else False
        has_formula = any((gcell(grow, c) or {}).get("formula") for c in value_cols)
        letters = [ch for ch in desc if ch.isalpha()]

        rows.append({
            "row":              r,
            "description":      desc,
            "has_values":       len(numeric) > 0,
            "value_count":      len(numeric),
            "is_bold":          is_bold,
            "total_keyword":    has_total_keyword(desc),
            "has_formula":      bool(has_formula),
            "whitespace_depth": len(a_raw) - len(a_raw.lstrip()),
            "column_depth":     0,                       # everything in column A
            "all_caps":         bool(letters) and all(ch.isupper() for ch in letters),
            "position_in_file": round(r / max_row, 2),
            "_last_value":      numeric[-1] if numeric else None,
        })

    _apply_arithmetic(rows)
    return rows

## Module 8 — Unknown route (AI structure map + arithmetic validation)

When format detection returns `unknown`, we trust the **AI structure agent** to read the raw grid and produce the full structure map — no deterministic candidate, no reconcile. The earlier deterministic `derive_layout` was removed: it was biased ("any numeric column is a value column") and it *overrode* the AI's correct reads (e.g. it re-added a `Variance` column the AI had correctly excluded). We now trust the AI map and **verify it with arithmetic**.

### 8a — AI structure-map agent
`run_structure_agent(cell_grid)` sends the raw grid to a capable model (`claude-opus-4-8`) and gets one JSON map: `description_columns`, code handling, `value_columns` (period columns only — not Variance/%/budget), `month_header_row`, `data_start_row`, the section boundaries (`revenue_end_row` / `expense_end_row` / `noi_row`), and the full **subsection hierarchy** (name / header_row / subtotal_row at every nesting level).

### 8b — Validate the AI map with arithmetic
`build_final_map` takes the AI map verbatim and **checks each subsection with math**: subtotal == sum of its non-subtotal leaf rows (nesting-correct). Only arithmetically-validated subtotals become the `is_subsection_subtotal` anchors used downstream. A quick boundary gate (`revenue − expense ≈ NOI`) confirms the map's totals are self-consistent. The map is trusted; math is the check — and the per-month NOI verification (Module 12) + feedback loop (Module 14) are the final safety nets.

### Module 8a — AI structure-map agent

`run_structure_agent(cell_grid)` reads the **raw grid** (no deterministic candidate) and returns the full `StructureMap`: layout (description/code/value columns, header + data-start rows), the section boundaries, and the subsection hierarchy. Model: `claude-opus-4-8`. `SHOW_PROMPT` prints the exact prompt.

In [28]:
import os
import sys

# make the t12_pipeline package importable + load its .env
_SRC = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "t12_pipeline" / "config" / "paths.py").is_file()), None)
if _SRC and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
if _SRC:
    from dotenv import load_dotenv
    load_dotenv(_SRC.parent / ".env")   # app/t12_pipeline/.env

from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from openpyxl.utils import column_index_from_string
from t12_pipeline.llm.llm_client import make_llm


# ── Module 8a — AI structure-map agent (reads the raw grid; the AI map is trusted)
def render_grid_for_agent(grid):
    """Fully-raw text view: each row 'row | COL=value  COL=value ...'.
    Text keeps its leading spaces; '*' marks bold; numbers rounded. No collapsing,
    no annotation — a capable model reads the raw grid directly."""
    L = [f"Sheet '{grid['sheet']}': used range rows 1-{grid['max_row']}, cols A-{get_column_letter(grid['max_col'])}."]
    if grid["merged_ranges"]:
        L.append(f"Merged cell ranges: {', '.join(grid['merged_ranges'])}")
    L.append("Each line below is one row: row_number | COL=cell_value  (text keeps its leading spaces; * = bold; numbers rounded).")
    L.append("")
    for grow in grid["rows"]:
        parts = []
        for c, cell in sorted(grow["cells"].items()):
            v = cell["value"]
            if v is None:
                continue
            star = "*" if cell["bold"] else ""
            v = round(v, 2) if isinstance(v, (int, float)) else repr(v)
            parts.append(f"{cell['letter']}{star}={v}")
        if parts:
            L.append(f"{grow['row']:>4} | " + "  ".join(parts))
    return "\n".join(L)


class Subsection(BaseModel):
    name: str = Field(description="the sub-group label as it reads in the sheet, e.g. 'Rental Income', 'Concessions', 'Payroll', 'Utilities'.")
    header_row: int | None = Field(description="the row where this sub-group STARTS — its header/label row, or its first line-item row if there is no separate header. Used to bound the sub-group for arithmetic checking.")
    subtotal_row: int = Field(description="the row holding this sub-group's subtotal (its value = the sum of the sub-group's line items).")


class StructureMap(BaseModel):
    # Layout — verify against the grid and correct if wrong
    description_columns: list[str] = Field(description="EVERY column letter that can hold an account description, left to right. In hierarchical/merged layouts a row's depth is shown by which column its text starts in — list all such columns.")
    code_column: str | None = Field(description="Column letter of a DEDICATED code column (codes only, separate from descriptions). If codes are embedded as a prefix inside the description text, set this to null.")
    code_embedded_in_description: bool = Field(description="True if account codes are a prefix inside the description text (e.g. '5120.0000 Rental Income'); False if codes have their own column or there are no codes.")
    code_pattern: str | None = Field(description="The code style, e.g. 'dot 5120.0000', 'colon 4010:', or null if there are no codes.")
    value_columns: list[str] = Field(description="ALL column letters holding the monthly numeric values (the full span).")
    month_header_row: int | None = Field(description="Row number of the month/period headers, or null.")
    data_start_row: int = Field(description="Row of the FIRST actual line-item (where the values begin).")
    # Sections — proposal, validated by arithmetic downstream
    revenue_end_row: int | None = Field(description="Row of the revenue grand total (Total Income / Effective Gross Income / Total Revenue).")
    expense_end_row: int | None = Field(description="Row of the operating-expense grand total (Total Operating Expenses / Total Expense).")
    noi_row: int | None = Field(description="Row of the Net Operating Income line.")
    subsections: list[Subsection] = Field(description="EVERY sub-group inside the revenue or expense sections that has its OWN subtotal row — at ALL nesting levels, top to bottom. Empty only if the sheet truly has no sub-group subtotals.")
    reasoning: str = Field(description="Brief explanation: note any field you CORRECTED from the candidate and why, and the labels used for the section rows.")


STRUCTURE_PROMPT = """You are an expert at reading commercial real estate T12 (trailing-12-month) income statements exported from various property-management systems.

You are given the raw cell grid of ONE worksheet. Identify its full structure so a deterministic parser can extract every row. Read the grid carefully — layouts vary widely (descriptions in one column or spread across several, different code styles, and extra columns such as Variance / % / budget that are NOT monthly periods).

Return the full structure:
- description_columns: EVERY column that can hold an account description, left to right (in hierarchical/merged layouts, depth is shown by which column the text starts in — list all of them).
- code_column / code_embedded_in_description / code_pattern: codes in their OWN dedicated column -> give that column, embedded=false. Codes as a prefix INSIDE the description text (e.g. '5120.0000 Rental Income') -> code_column=null, embedded=true. Describe the style in code_pattern.
- value_columns: ALL columns holding the monthly PERIOD numbers (the full span). Include a trailing total column (e.g. 'Total' / 'Adjusted Total') only if it is a genuine sum of the months; do NOT include derived columns such as 'Variance', '%', or a budget column.
- month_header_row: the row with the month/period headers.
- data_start_row: the first actual line-item row.
- revenue_end_row / expense_end_row / noi_row: the revenue grand total (Total Income / EGI), the operating-expense grand total (Total Operating Expenses), and the Net Operating Income line. These are a PROPOSAL — checked downstream against the arithmetic (revenue_total - expense_total ~= NOI at the last value column).
- subsections: EVERY sub-group inside the revenue or expense sections that has its OWN subtotal row, at ALL nesting levels, top to bottom. For each, give its name, its header/start row (header_row), and its subtotal row. A sub-group's subtotal is the row whose value equals the sum of that sub-group's line items. CRITICAL — a total row is NOT always obvious. It may be marked ANY of these ways, and you must include ALL of them: (a) the word 'Total' in its text; (b) BOLD 'repeated-name' text without the word 'Total' (e.g. a bold `Payroll` row summing the payroll lines); (c) ONLY by its account code with no descriptive text and no bold — e.g. a code ending in a round/rollup suffix such as ...099 or ...999 (like `41999-099 Total Rental Inc.` or `49999-999 Total Income`); or (d) simply by being the arithmetic sum of the plain detail rows above it. Deep hierarchies stack several of these; missing even one makes the downstream arithmetic double-count. When unsure whether a value row is a line-item or a total, check: does its value equal the sum of the rows above it? If yes, it is a total.

Use the exact row numbers and column letters shown in the grid.

Cell grid:
{grid_text}
"""

AGENT_MODEL = "claude-opus-4-8"   # Rung-2 reasoning model (Opus) for structure mapping


def build_structure_prompt(grid):
    """Return the EXACT prompt string that will be sent to the LLM (for inspection)."""
    return STRUCTURE_PROMPT.format(grid_text=render_grid_for_agent(grid))


def run_structure_agent(grid):
    """Chain built the src way; returns a full StructureMap from the raw grid."""
    llm = make_llm(provider="anthropic", model=AGENT_MODEL, temperature=0)
    structured_llm = llm.with_structured_output(StructureMap)
    prompt = PromptTemplate(input_variables=["grid_text"], template=STRUCTURE_PROMPT)
    chain = prompt | structured_llm
    return chain.invoke({"grid_text": render_grid_for_agent(grid)})


def _last_col_value(grid, row_num, last_col_letter):
    """Value at (row_num, last value column) — used by the arithmetic gate in 8c."""
    if row_num is None:
        return None
    grow = grid_row_map(grid).get(row_num)
    return gvalue(grow, column_index_from_string(last_col_letter)) if grow else None


SHOW_PROMPT = True   # print the exact prompt sent to the LLM before invoking

### Module 8b — Validate the AI map (arithmetic; no reconcile)

`build_final_map(ai_map, grid)` trusts the AI map and **verifies it with math**: each subsection subtotal must equal the sum of its leaf line-items (nesting-correct); only validated subtotals become anchors. Plus a boundary gate (`revenue − expense ≈ NOI`). No deterministic override — the AI decides the structure, arithmetic decides if it's consistent.

In [29]:
# ── Module 8b — Validate the AI structure map with arithmetic (no reconcile) ──
# We trust the AI's structure map directly. This step CHECKS it with math: each
# subsection subtotal must equal the sum of its leaf line-items. Only arithmetically
# validated subtotals become anchors downstream.


def validate_subsections(final, grid):
    """Arithmetic-check each AI subsection: subtotal == sum of the leaf value rows in
    its [header_row, subtotal_row) span that are NOT themselves subtotal rows.
    Summing only non-subtotal leaves makes this correct for nested roll-ups too.
    Returns (per-subsection results, set of VALIDATED subtotal rows)."""
    last_i = column_index_from_string(final["value_columns"][-1])
    rmap   = grid_row_map(grid)
    def cval(r):
        g = rmap.get(r); x = gvalue(g, last_i) if g else None
        return x if isinstance(x, (int, float)) else None

    subs = final["subsections"]
    subtotal_set = {s["subtotal_row"] for s in subs}
    rev_end = final["revenue_end_row"]; data_start = final["data_start_row"]

    results, validated = [], []
    for s in subs:
        R = s["subtotal_row"]; H = s["header_row"]; target = cval(R)
        # span start: the AI header_row if it sits inside this section, else the section start
        sec_start = data_start if (rev_end is None or R <= rev_end) else rev_end + 1
        start = H if (isinstance(H, int) and sec_start <= H < R) else sec_start
        members = [rr for rr in range(start, R)
                   if rr not in subtotal_set and cval(rr) is not None]
        msum = sum(cval(rr) for rr in members)
        ok = target is not None and abs(msum - target) <= max(1.0, abs(target) * 0.005)
        results.append({"name": s["name"], "row": R, "header_row": H, "span": [start, R],
                        "member_rows": members, "member_values": [round(cval(rr), 2) for rr in members],
                        "members": len(members), "sum": round(msum, 2),
                        "target": round(target, 2) if isinstance(target, (int, float)) else target,
                        "ok": ok})
        if ok:
            validated.append(R)
    return results, sorted(set(validated))


def build_final_map(ai_map, grid):
    """Turn the AI structure map into the final_map used downstream: the AI's fields
    verbatim, PLUS the arithmetically-validated subsection anchors. No deterministic
    override — the AI map is trusted; math only validates the subsections.
    Returns (final_map, per-subsection results)."""
    final = ai_map if isinstance(ai_map, dict) else ai_map.model_dump()
    sub_results, validated_rows = validate_subsections(final, grid)
    boundary = {final["revenue_end_row"], final["expense_end_row"], final["noi_row"]} - {None}
    final["subsection_subtotal_rows"]           = sorted({s["subtotal_row"] for s in final["subsections"]} - boundary)
    final["validated_subsection_subtotal_rows"] = [r for r in validated_rows if r not in boundary]
    final["subsection_header_rows"]             = sorted({s["header_row"] for s in final["subsections"]
                                                          if isinstance(s["header_row"], int)} - boundary)
    return final, sub_results


## Module 9 — Dynamic signal extraction (unknown route)

The counterpart to Modules 5–7, but driven by the **final structure map** (8c) instead of a hard-coded per-format layout. It walks `cell_grid` and emits the **exact same per-row feature vector** the known extractors produce, so the unknown route ends with identical output for the same downstream classifier.

How it reads the map per row:
- **description** — the first non-empty cell scanning `description_columns` left-to-right; the embedded code prefix is stripped (a dedicated `code_column`, if any, is already excluded from the description columns).
- **`column_depth`** — the index of that description column within `description_columns`. In a **TREE** layout the hierarchy lives here (deeper rows start further right; subtotals snap back to column A = depth `0`); in a **FLAT** layout it stays `0` and the depth lives in `whitespace_depth`.
- **values** — from `value_columns`; **`arithmetic_sum_of_above`** via the shared `_apply_arithmetic` (same trust anchor as the known routes).

Runs only for `unknown` files, after the structure map is built.

In [30]:
# ── Module 9 — Dynamic signal extraction driven by the final structure map ────
# Emits the per-row feature vector — the RAW SIGNAL POOL the classifier scores —
# plus the raw monthly `values` (12 months + total). column_depth carries the TREE
# hierarchy; whitespace_depth carries it for FLAT layouts. The AI subsection anchors
# (is_subsection_subtotal / is_subsection_header) are kept as FACTS for a later
# override layer — the classifier does NOT score them.
import json

# strip a leading account code of any known style (dot / colon / mri / 5-digit / dash)
CODE_PREFIX_RE = re.compile(
    r"^\s*(?:\d{4}\.\d+|\d{4}:|\d-\d{4}-\d+|\d{5}-\d+|\d{4}\s*-\s*\d*\s*-?)\s*"
)

# COMPUTED_METRIC keyword pool — a figure computed ACROSS sections (revenue − expense).
# EGI / "total income" are revenue grand totals, NOT metrics.
METRIC_KEYWORDS = ["net operating income", "noi", "net cash flow",
                   "income after debt service", "cash flow after"]


def is_metric(desc):
    d = (desc or "").strip().lower()
    return any(k in d for k in METRIC_KEYWORDS)


def _desc_from_columns(grow, desc_cols_idx):
    """First non-empty text cell scanning the description columns left-to-right.
    Returns (raw_text, depth_index) or (None, None)."""
    for depth, c in enumerate(desc_cols_idx):
        v = gvalue(grow, c)
        if isinstance(v, str) and v.strip():
            return v, depth
    return None, None


def extract_row_signals_dynamic(grid, final_map):
    """Per-row feature vector for an unknown-format sheet, driven by final_map.
    Same schema as Modules 5-7 + monthly values + the raw signal pool."""
    max_row = grid["max_row"]
    desc_cols_idx = [column_index_from_string(c) for c in final_map["description_columns"]]
    value_cols    = [column_index_from_string(c) for c in final_map["value_columns"]]
    embedded      = final_map["code_embedded_in_description"]
    month_hdr     = final_map.get("month_header_row")
    # anchors from the AI's arithmetic-validated subsection hierarchy — kept as FACTS
    # (a later override layer may use them); the classifier does NOT score them.
    sub_subtotals = set(final_map.get("validated_subsection_subtotal_rows",
                                      final_map.get("subsection_subtotal_rows", [])))
    sub_headers   = set(final_map.get("subsection_header_rows", []))
    rows = []

    # ── pass 1 ──
    for grow in grid["rows"]:
        r = grow["row"]
        raw, depth = _desc_from_columns(grow, desc_cols_idx)
        raw = raw if raw is not None else ""
        desc = CODE_PREFIX_RE.sub("", raw).strip() if embedded else raw.strip()

        raw_vals = [gvalue(grow, c) for c in value_cols]                 # per value-column
        numeric  = [v for v in raw_vals if isinstance(v, (int, float))]
        if not desc and not numeric:
            continue

        desc_col = desc_cols_idx[depth] if depth is not None else desc_cols_idx[0]
        desc_cell = gcell(grow, desc_col)
        is_bold = bool(desc_cell["bold"]) if desc_cell else False
        has_formula = any((gcell(grow, c) or {}).get("formula") for c in value_cols)
        # accountant's sum-line: a top border on the label cell or any value cell
        top_border = bool(desc_cell and desc_cell.get("border_top")) or \
                     any((gcell(grow, c) or {}).get("border_top") for c in value_cols)
        letters = [ch for ch in desc if ch.isalpha()]

        # ── provenance: WHY each signal is what it is (for logging / client explain) ──
        code_m = CODE_PREFIX_RE.match(raw) if embedded else None
        border_cells  = [get_column_letter(c) for c in [desc_col, *value_cols]
                         if (gcell(grow, c) or {}).get("border_top")]
        formula_cells = {get_column_letter(c): (gcell(grow, c) or {}).get("formula")
                         for c in value_cols if (gcell(grow, c) or {}).get("formula")}
        tk_hit = total_keyword_hit(desc)
        mk_hit = next((k for k in METRIC_KEYWORDS if k in desc.lower()), None)

        rows.append({
            "row":                    r,
            "description":            desc,
            "has_values":             len(numeric) > 0,
            "value_count":            len(numeric),
            "values":                 [round(v, 2) if isinstance(v, (int, float)) else None
                                       for v in raw_vals],           # 12 months + total, aligned to value_columns
            "is_bold":                is_bold,
            "total_keyword":          has_total_keyword(desc),
            "is_metric":              is_metric(desc),
            "has_formula":            bool(has_formula),
            "top_border":             bool(top_border),
            "whitespace_depth":       len(raw) - len(raw.lstrip()),
            "column_depth":           depth if depth is not None else 0,
            "all_caps":               bool(letters) and all(ch.isupper() for ch in letters),
            "position_in_file":       round(r / max_row, 2),
            "in_metadata_band":       month_hdr is not None and r <= month_hdr,
            "is_subsection_subtotal": r in sub_subtotals,
            "is_subsection_header":   r in sub_headers,
            "signal_evidence": {
                "raw_description":    raw,
                "stripped_code":      code_m.group(0).strip() if (code_m and code_m.group(0).strip()) else None,
                "desc_column":        get_column_letter(desc_col),
                "leading_spaces":     len(raw) - len(raw.lstrip()),
                "bold_cell":          get_column_letter(desc_col) if is_bold else None,
                "total_keyword_hit":  tk_hit,
                "metric_keyword_hit": mk_hit,
                "subtotal_marked_by": [n for n, on in (("bold", is_bold), ("total_keyword", bool(tk_hit))) if on],
                "top_border_cells":   border_cells,
                "formula_cells":      formula_cells,
            },
            "_last_value":            numeric[-1] if numeric else None,
        })

    _apply_arithmetic(rows)
    return rows


In [ ]:
# ── Module 8c — Stage 1 Anchor Lock (R − E = N arithmetic decides) ────────────
# Nominate candidate R / E / NOI rows (keywords from anchor_keywords.json UNION the
# AI's proposed rows), then let the month-by-month R − E = N arithmetic pick the real
# triple. Exactly one positionally-coherent passing triple auto-locks; zero/multiple
# → gate (bottom-most tie-break for batch). Keywords only NOMINATE; the math DECIDES.
# This replaces "trust the AI boundary" + the keyword state machine, and it kills the
# after-debt keyword collision (that row misses R−E=N by the debt service).
import json as _json
from pathlib import Path as _Path


def load_anchor_keywords(path=None):
    p = _Path(path) if path else (_Path.cwd() / "anchor_keywords.json")
    with open(p) as f:
        return _json.load(f)


ANCHOR_KEYWORDS = load_anchor_keywords()

_ATOL, _AFRAC, _MINCOLS = 1.0, 0.005, 6      # tolerance max($1, 0.5%); >=6 month columns


def _mvals(row):
    """Per-value-column values of a row (numeric or None), aligned to value_columns."""
    return [v if isinstance(v, (int, float)) else None for v in row.get("values", [])]


def _ncount(vals):
    return sum(1 for v in vals if isinstance(v, (int, float)))


def _amatch(a, c):
    return a is not None and c is not None and abs(a - c) <= max(_ATOL, abs(a) * _AFRAC)


def _hits(rows, markers, exclusions):
    out = []
    for r in rows:
        d = r["description"].strip().lower()
        if any(x in d for x in exclusions):
            continue
        if any(m in d for m in markers) and _ncount(_mvals(r)) >= _MINCOLS:
            out.append(r["row"])
    return out


def nominate_anchors(signal_rows, kw, ai_rows):
    """Per anchor type: grand-marker rows (intermediates only if no grand fires),
    minus exclusions, requiring >=6 numeric month values; unioned with the AI's rows."""
    cands = {}
    for t in ("revenue", "expense", "noi"):
        cfg = kw[t]
        c = _hits(signal_rows, cfg["grand_markers"], cfg.get("exclusions", []))
        if not c:
            c = _hits(signal_rows, cfg.get("intermediate_markers", []), cfg.get("exclusions", []))
        if ai_rows.get(t):
            c = sorted(set(c) | {ai_rows[t]})
        cands[t] = sorted(set(c))
    return cands


def _expense_units(signal_rows, kw, exp_cands):
    """Expense candidates as single rows PLUS split-pair sums (controllable + non-)."""
    units = [[e] for e in exp_cands]
    for pair in kw["expense"].get("split_pair_markers", {}).get("pairs", []):
        legs = []
        for m in pair:
            legs.append([r["row"] for r in signal_rows
                         if m in r["description"].strip().lower() and _ncount(_mvals(r)) >= _MINCOLS])
        for a in legs[0]:
            for b in legs[1]:
                if a != b:
                    units.append(sorted([a, b]))
    seen, out = set(), []
    for u in units:
        k = tuple(u)
        if k not in seen:
            seen.add(k); out.append(u)
    return out


def _triple_holds(rrow, eunit, nrow, byrow):
    rv = _mvals(byrow[rrow]); nv = _mvals(byrow[nrow])
    n = len(rv)
    ev = [None] * n
    for er in eunit:
        for i, x in enumerate(_mvals(byrow[er])):
            if isinstance(x, (int, float)):
                ev[i] = (ev[i] or 0.0) + x
    matched = 0
    for i in range(n):
        if rv[i] is None or ev[i] is None or nv[i] is None:
            continue
        if not _amatch(nv[i], rv[i] - ev[i]):
            return False
        matched += 1
    return matched >= _MINCOLS


def lock_anchors(signal_rows, final_map):
    """Nominate → triple-test → lock. Returns dict(status, method, anchors, passing,
    candidates). status ∈ {AUTO, GATE, NONE}. `anchors` gives revenue_end_row /
    expense_end_row / noi_row (expense boundary = bottom-most of a split pair)."""
    byrow = {r["row"]: r for r in signal_rows}
    ai_rows = {"revenue": final_map.get("revenue_end_row"),
               "expense": final_map.get("expense_end_row"),
               "noi": final_map.get("noi_row")}
    cands = nominate_anchors(signal_rows, ANCHOR_KEYWORDS, ai_rows)
    eunits = _expense_units(signal_rows, ANCHOR_KEYWORDS, cands["expense"])

    passing = []
    for rr in cands["revenue"]:
        for eu in eunits:
            if rr in eu:
                continue
            for nr in cands["noi"]:
                if nr == rr or nr in eu:
                    continue
                if _triple_holds(rr, eu, nr, byrow):
                    passing.append((rr, eu, nr))

    coherent = [(rr, eu, nr) for (rr, eu, nr) in passing if rr < min(eu) and max(eu) < nr]
    pool = coherent or passing

    def _pack(t):
        rr, eu, nr = t
        return {"revenue_end_row": rr, "expense_end_row": max(eu), "noi_row": nr,
                "expense_split_rows": (eu if len(eu) > 1 else None)}

    if len(pool) == 1:
        return {"status": "AUTO", "method": "auto", "anchors": _pack(pool[0]),
                "passing": pool, "candidates": cands}
    if len(pool) > 1:
        # bottom-most per anchor: the grand total is the LAST total of its kind
        best = max(pool, key=lambda t: (t[0], max(t[1]), t[2]))
        return {"status": "GATE", "method": "manual", "anchors": _pack(best),
                "passing": pool, "candidates": cands}
    return {"status": "NONE", "method": None, "anchors": None,
            "passing": [], "candidates": cands}


def apply_anchor_overrides(classified, final_map):
    """Arithmetic-verified facts OVERRIDE the classifier score: the locked anchors and
    the arithmetically-validated subsection subtotals get their class forced."""
    facts = {}
    if final_map.get("noi_row"):
        facts[final_map["noi_row"]] = "COMPUTED_METRIC"
    for b in (final_map.get("revenue_end_row"), final_map.get("expense_end_row")):
        if isinstance(b, int):
            facts[b] = "SECTION_SUBTOTAL"
    for s in final_map.get("validated_subsection_subtotal_rows", []):
        facts.setdefault(s, "SUBSECTION_SUBTOTAL")
    out = []
    for r in classified:
        r2 = dict(r)
        if r["row"] in facts:
            r2["class"] = facts[r["row"]]
        out.append(r2)
    return out


## Module 10 — Row classifier (score-based, 7 classes)

Turns each feature vector (Modules 5–7 / 9) into one of seven row types. **Shared by every route** — it only reads the feature vector plus a few anchors from the structure map, so known and unknown files classify the same way.

**Why score-based** (not a decision tree or an ML model): every row is scored for *all seven* classes and the highest wins — so the decision is **transparent** (you see the runner-up and the margin), **tunable** (adjust one weight, not retrain), and **degrades gracefully** (a weird row still lands on its best-fit class instead of falling through a tree branch). Printing all seven scores is the audit trail.

The classes split on two axes — **has values?** and **depth / total-ness**:

| Class | Core signature |
|---|---|
| `METADATA` | no values, at/above `month_header_row` (title / address / date / column header) |
| `SECTION_HEADER` | no values, shallow (depth ≤ 1), bold/caps, in the body (`INCOME`) |
| `SUBSECTION_HEADER` | no values, deep (depth ≥ 2), bold (`Rental Income`) |
| `DATA` | has values, **plain (not bold)**, not "Total", not a metric |
| `SUBSECTION_SUBTOTAL` | has values, **bold** and/or `"Total …"` and/or `arithmetic_sum_of_above`, not caps |
| `SECTION_SUBTOTAL` | has values, grand total (often ALL-CAPS), **at a section-boundary row** |
| `COMPUTED_METRIC` | metric keyword (NOI / net cash flow / after debt service) or at `noi_row` |

**The key discriminators.** `month_header_row` separates the metadata band from the first section header (which sits just below it, *before* the first valued row). The arithmetic-validated **section-boundary rows** tell a *grand* total (`TOTAL INCOME`) from a *subsection* total. And for value rows, **`is_bold` is a primary subtotal signal** — leaf data is plain, subtotals are bold. This is what catches flat **"repeated-name" subtotals** (e.g. a bold `Payroll` row totalling the payroll lines) that carry no "Total" prefix and on which arithmetic can't fire because the un-flagged nested subtotals pollute the running sum.

**Correctness self-check.** After classifying, the module runs a **leaf-DATA reconciliation**: the values of *only* the `DATA`-class rows in each section must sum to that section's grand total. If a subtotal is misclassified as data it double-counts; if data is misclassified as a subtotal it under-counts — either way the sum breaks. Passing to the penny on both a nested-TREE (Cardinal) and a flat (Holden) format is strong evidence the classes are right, and it's the gate that flagged the "repeated-name subtotal" bug in the first place.

Runs on the feature vectors from whichever extractor produced them (here: the unknown-route Module 9 output).

In [31]:
# ── Module 10 — Normalized signal-match classifier (7 classes) ────────────────
# UNBIASED scoring: no hand-tuned weights. Each class lists the signals it should
# satisfy (CLASS_SIGNALS); a row's score for a class = the FRACTION of that class's
# signals it satisfies (satisfied / total). argmax wins. Normalising by the signal
# count lets classes with different numbers of signals compare fairly (3/3 beats
# 3/8). Contra-indicators are expressed as positive signals (not_bold,
# no_total_keyword) so an off-class row FAILS them and its fraction drops. Adding a
# signal = add its name to a class list (+ its evaluator in SIGNAL_FUNCS); both dicts
# are config, meant to move to a config file later. The AI subsection anchors are
# deliberately NOT scored here — they are facts for a later override layer.
from collections import Counter

CLASSES = ["METADATA", "SECTION_HEADER", "SUBSECTION_HEADER", "DATA",
           "SUBSECTION_SUBTOTAL", "SECTION_SUBTOTAL", "COMPUTED_METRIC"]
_SHORT = ["META", "SECHDR", "SUBHDR", "DATA", "SUBTOT", "SECTOT", "METRIC"]

# WHICH signals each class considers (config). has_values / no_values splits the
# value rows from the label rows automatically; the rest discriminate within a group.
# DESIGN RULE: a class lists only signals that are RELIABLY true for it — a
# sometimes-present signal would unfairly lower the fraction of members that lack it.
# `subtotal_marked` (bold OR a "Total" keyword) is the reliable "this row is a total"
# cue: across our corpus NO data row is bold, and every non-bold subtotal carries a
# total keyword. Folding both into one signal stops a single-cue subtotal (Holden's
# bold-only rows, Eleven11's total-keyword-only rows) from being diluted to a low
# fraction. Rarer cues (arithmetic / top_border / has_formula) are left in the pool
# (see SIGNAL_FUNCS) but not REQUIRED here; the feedback loop catches the invisible
# rollups they would target.
CLASS_SIGNALS = {
    "METADATA":            ["no_values", "in_metadata_band", "no_total_keyword"],
    "SECTION_HEADER":      ["no_values", "in_body", "bold_or_caps", "shallow"],
    "SUBSECTION_HEADER":   ["no_values", "in_body", "is_bold", "deep"],
    "DATA":                ["has_values", "not_subtotal_marked", "not_metric"],
    "SUBSECTION_SUBTOTAL": ["has_values", "in_body", "subtotal_marked",
                            "not_metric", "not_all_caps"],
    "SECTION_SUBTOTAL":    ["has_values", "subtotal_marked", "all_caps", "not_metric"],
    "COMPUTED_METRIC":     ["has_values", "in_body", "is_metric"],
}

# HOW each signal name is evaluated on a feature-vector row (config). Negated and
# derived signals live here, so the extractor only stores raw booleans.
SIGNAL_FUNCS = {
    "has_values":              lambda r: bool(r["has_values"]),
    "no_values":               lambda r: not r["has_values"],
    "is_bold":                 lambda r: bool(r["is_bold"]),
    "not_bold":                lambda r: not r["is_bold"],
    "all_caps":                lambda r: bool(r["all_caps"]),
    "not_all_caps":            lambda r: not r["all_caps"],
    "bold_or_caps":            lambda r: bool(r["is_bold"] or r["all_caps"]),
    "total_keyword":           lambda r: bool(r["total_keyword"]),
    "no_total_keyword":        lambda r: not r["total_keyword"],
    "subtotal_marked":         lambda r: bool(r["is_bold"] or r["total_keyword"]),
    "not_subtotal_marked":     lambda r: not (r["is_bold"] or r["total_keyword"]),
    "is_metric":               lambda r: bool(r["is_metric"]),
    "not_metric":              lambda r: not r["is_metric"],
    "arithmetic_sum_of_above": lambda r: bool(r.get("arithmetic_sum_of_above")),
    "no_arithmetic":           lambda r: not r.get("arithmetic_sum_of_above"),
    "has_formula":             lambda r: bool(r["has_formula"]),
    "top_border":              lambda r: bool(r["top_border"]),
    "no_top_border":           lambda r: not r["top_border"],
    "deep":                    lambda r: r["column_depth"] >= 2 or r["whitespace_depth"] >= 2,
    "shallow":                 lambda r: r["column_depth"] <= 1 and r["whitespace_depth"] < 2,
    "in_metadata_band":        lambda r: bool(r.get("in_metadata_band")),
    "in_body":                 lambda r: not r.get("in_metadata_band"),
}


def score_row(f):
    """Score against all 7 classes = fraction of each class's signals satisfied.
    Unbiased: every signal counts 1; normalised by the class's signal count."""
    return {cls: round(sum(SIGNAL_FUNCS[s](f) for s in sigs) / len(sigs), 3)
            for cls, sigs in CLASS_SIGNALS.items()}


def classify_rows(rows, structure_map=None):
    """Attach `class` (argmax of the normalised fractions) and `scores` to each row.
    structure_map is accepted for call-site compatibility but unused — every signal
    is already baked into the feature vector by the extractor. Ties break toward the
    earlier class in CLASSES (dict-insertion order)."""
    out = []
    for f in rows:
        sc = score_row(f)
        out.append({**f, "class": max(sc, key=sc.get), "scores": sc})
    return out


def print_classified_rows(rows, limit=None):
    """Row, description, chosen class, and ALL seven class fractions (audit trail)."""
    hdr = f"{'row':>4}  {'description':<28}  {'CLASS':<20} " + " ".join(f"{s:>6}" for s in _SHORT)
    print(hdr); print("-" * len(hdr))
    for r in (rows if limit is None else rows[:limit]):
        sc = r["scores"]
        print(f"{r['row']:>4}  {r['description'][:28]:<28}  {r['class']:<20} "
              + " ".join(f"{sc[c]:>6.2f}" for c in CLASSES))


def _sig_label(s, r):
    """A signal name annotated with the provenance behind it (which cue / keyword)."""
    ev = r.get("signal_evidence", {})
    if s in ("subtotal_marked", "not_subtotal_marked"):
        by = ev.get("subtotal_marked_by") or []
        return f"{s}({'+'.join(by) or 'none'})"
    if s == "total_keyword" and ev.get("total_keyword_hit"):
        return f"total_keyword('{ev['total_keyword_hit']}')"
    if s == "is_metric" and ev.get("metric_keyword_hit"):
        return f"is_metric('{ev['metric_keyword_hit']}')"
    return s


def explain_scores(rows, structure_map=None, example_rows=None, top_k=2, which="examples"):
    """Show the winning class and runner-up with the exact signals that fired
    (hit / total), each annotated with its provenance. `which`: 'examples' = one row
    per class (default), 'all' = every row, or pass explicit row numbers via
    example_rows."""
    by_row = {r["row"]: r for r in rows}
    if example_rows is None:
        if which == "all":
            example_rows = [r["row"] for r in rows]
        else:
            seen = {}
            for r in rows:
                seen.setdefault(r["class"], r["row"])
            example_rows = list(seen.values())
    for rn in example_rows:
        r = by_row.get(rn)
        if r is None:
            continue
        sc = r["scores"]
        ranked = sorted(CLASSES, key=lambda c: sc[c], reverse=True)
        print(f"\nrow {r['row']}  {r['description'][:44]!r}  →  {r['class']}  ({sc[r['class']]:.0%})")
        for c in ranked[:top_k]:
            sigs = CLASS_SIGNALS[c]
            hit  = [_sig_label(s, r) for s in sigs if SIGNAL_FUNCS[s](r)]
            miss = [_sig_label(s, r) for s in sigs if not SIGNAL_FUNCS[s](r)]
            tag = "WINNER " if c == ranked[0] else "runner "
            print(f"    {tag}{c:<20} {len(hit)}/{len(sigs)} = {sc[c]:.0%}   "
                  f"hit[{', '.join(hit) or '—'}]  miss[{', '.join(miss) or '—'}]")


def print_arithmetic_evidence(rows, only_fired=True, limit=None):
    """The `arithmetic_sum_of_above` audit trail: for each qualifying row, exactly
    which rows were summed, the sum, the row's own target value, and diff/tolerance —
    so a Y/N is never a black box. only_fired=True shows just the matches."""
    shown = [r for r in rows if r.get("arithmetic", {}).get("member_rows")
             and (r["arithmetic"]["fired"] or not only_fired)]
    if not shown:
        print("  (no rows matched an arithmetic sum-of-above)")
        return
    for r in (shown if limit is None else shown[:limit]):
        a = r["arithmetic"]; mr = a["member_rows"]
        rng = f"{mr[0]}..{mr[-1]}" if len(mr) > 1 else f"{mr[0]}"
        mark = "✓" if a["fired"] else "✗"
        print(f"  row {r['row']:>3} {r['description'][:24]:<24} = Σ rows {rng:<10} ({len(mr):>2}) "
              f"= {a['sum']:>15,.2f} vs target {a['target']:>15,.2f}  diff {a['diff']:>10,.2f} / tol {a['tolerance']:>8,.2f}  {mark}")
        if len(a["member_values"]) <= 8:
            print(f"        values: {' + '.join(f'{x:,.2f}' for x in a['member_values'])}")


def reconcile_leaf_data(rows, grid, structure_map):
    """Correctness self-check: the values of ONLY the DATA-class rows in each section
    must sum to that section's grand total. Returns a dict with the computed vs
    vendor figure, pass/fail, AND the exact DATA rows summed (so a mismatch is
    traceable to specific rows)."""
    last  = column_index_from_string(structure_map["value_columns"][-1])
    rmap  = grid_row_map(grid)
    def v(r):
        g = rmap.get(r); x = gvalue(g, last) if g else None
        return x if isinstance(x, (int, float)) else 0.0
    mh, rev_end, exp_end, noi_r = (structure_map["month_header_row"], structure_map["revenue_end_row"],
                                   structure_map["expense_end_row"], structure_map["noi_row"])
    rev_rows = [r["row"] for r in rows if r["class"] == "DATA" and mh < r["row"] < rev_end]
    exp_rows = [r["row"] for r in rows if r["class"] == "DATA" and rev_end < r["row"] < exp_end]
    rev = sum(v(rr) for rr in rev_rows); exp = sum(v(rr) for rr in exp_rows)
    ok = lambda a, b: abs(a - b) <= max(1.0, abs(b) * 0.005)
    return {
        "revenue": {"computed": round(rev, 2), "vendor": round(v(rev_end), 2),
                    "ok": ok(rev, v(rev_end)), "rows": rev_rows, "n": len(rev_rows)},
        "expense": {"computed": round(exp, 2), "vendor": round(v(exp_end), 2),
                    "ok": ok(exp, v(exp_end)), "rows": exp_rows, "n": len(exp_rows)},
        "noi":     {"computed": round(rev - exp, 2), "vendor": round(v(noi_r), 2),
                    "ok": ok(rev - exp, v(noi_r))},
    }


## Module 11 — Section assignment (deterministic + AI, cross-verified)

Splits the old combined module in two. This one assigns sections; NOI verification moves to Module 12.

**Two INDEPENDENT branches determine the section boundaries, in parallel:**
- **Branch A — deterministic keyword match** (mirrors src `annotate_totals` + `keywords.py`): the bottom-most value row whose description matches a grand-total marker (`total income` / `total expense` / `net operating income`). Pure regex on descriptions — no AI, no classifier classes.
- **Branch B — the AI structure agent's boundaries** (semantic reading, from 8b).

If they **agree → HIGH confidence**. If they **disagree → route to review**, and the per-month NOI check in Module 12 is the arithmetic tie-breaker. This is a genuine triangulation: keyword ⟂ AI ⟂ arithmetic — three independent signals. (A *second AI call* would not be independent — same model, same grid → correlated errors, so agreement wouldn't mean much.)

**Section assignment** then runs a two-flag state machine: `revenue → expense` (after the revenue grand total) `→ capex` (after NOI). Boundary/subtotal rows keep the state *before* their flip; rows at/above `month_header_row` are `metadata`.


In [32]:
# ── Module 11 — Section assignment (deterministic + AI, cross-verified) ───────
# Two INDEPENDENT determinations of the section boundaries, run in parallel:
#   Branch A (deterministic): keyword match on value rows (src annotate_totals
#     markers) -> bottom-most match that carries values. Pure regex, no AI.
#   Branch B (AI): the boundaries the structure agent already returned (8b).
# AGREE -> HIGH confidence. Disagree -> route to review; the per-month NOI check
# in Module 12 is the arithmetic tie-breaker (the independent third signal).
# Then a two-flag state machine tags every row revenue -> expense -> capex.

# grand-total keyword markers (from src keywords.py — the independent signal)
GRAND_INCOME_MARKERS  = {"total income", "total revenue", "effective gross income",
                         "total operating revenues"}
GRAND_EXPENSE_MARKERS = {"total expenses", "total operating expenses",
                         "total controllable expenses", "total expense", "total operating costs"}
GRAND_NOI_MARKERS     = {"net operating income"}


def _kw_match(desc, markers):
    d = desc.lower()
    return any(m in d for m in markers)


def detect_boundaries_by_keyword(classified):
    """Branch A — independent, deterministic boundary finder (mirrors src
    annotate_totals): among rows that carry values, keep the bottom-most grand-total
    keyword match. Uses only descriptions + values — no AI, no classifier classes.
    Also records, per boundary, the matching row's description and which marker fired
    (under 'evidence') so the deterministic branch can be explained, not just trusted."""
    ev = {}
    def bottom_with_values(field, markers, before=None):
        hits = [r for r in classified
                if r["has_values"] and _kw_match(r["description"], markers)
                and (before is None or r["row"] < before)]
        if not hits:
            ev[field] = None
            return None
        h = hits[-1]
        ev[field] = {"row": h["row"], "desc": h["description"],
                     "matched_marker": next((m for m in markers if m in h["description"].lower()), None)}
        return h["row"]
    noi = bottom_with_values("noi_row", GRAND_NOI_MARKERS)
    rev = bottom_with_values("revenue_end_row", GRAND_INCOME_MARKERS, before=noi)
    exp = bottom_with_values("expense_end_row", GRAND_EXPENSE_MARKERS, before=noi)   # last expense total before NOI
    return {"revenue_end_row": rev, "expense_end_row": exp, "noi_row": noi, "evidence": ev}


def cross_verify_boundaries(ai_map, kw):
    """Compare the AI boundaries with the keyword boundaries field-by-field."""
    fields = ["revenue_end_row", "expense_end_row", "noi_row"]
    per = {f: {"ai": ai_map[f], "keyword": kw[f], "agree": ai_map[f] == kw[f]} for f in fields}
    return all(per[f]["agree"] for f in fields), per


def assign_sections(classified, final_map):
    """Two-flag state machine: revenue -> expense (after revenue grand total) ->
    capex (after NOI). Boundary rows keep the state BEFORE their flip (src convention)."""
    mh, rev_end, noi_row = (final_map["month_header_row"],
                            final_map["revenue_end_row"], final_map["noi_row"])
    state = "revenue"
    for r in classified:
        row = r["row"]
        if mh is not None and row <= mh:
            r["section"] = "metadata"; continue
        r["section"] = state
        if row == rev_end:
            state = "expense"
        elif row == noi_row:
            state = "capex"
    return classified

## Module 12 — NOI verification (bottom-up vs top-down, per month + total)

The arithmetic gate and the third independent signal in the triangulation. Mirrors src `compute_noi` + `compare_noi`:
- **bottom-up (our row method):** sum the `DATA`-class rows by assigned `section`; NOI = revenue − expense. Capex excluded.
- **top-down (vendor-reported):** read the grand-total rows directly.

Compared **per month AND on the total** — errors that cancel across months are still caught. `overall_pass` requires the bottom-up NOI to match the vendor NOI on **every** column. If Module 11's two branches disagreed, this is what decides which boundary set is actually correct (whichever makes the sums reconcile).


In [33]:
# ── Module 12 — NOI verification (bottom-up vs top-down, per month + total) ────
# The arithmetic tie-breaker / final gate. Mirrors src compute_noi + compare_noi:
#   bottom-up  = sum DATA rows by assigned section (capex excluded)
#   top-down   = the vendor's grand-total rows
# compared per-month AND on the total. overall_pass requires the bottom-up NOI to
# match the vendor NOI on EVERY month column and the total.

def _row_values(grid, row_num, value_cols):
    grow = grid_row_map(grid).get(row_num)
    return [gvalue(grow, c) if grow else None for c in value_cols]


def verify_noi(classified, grid, final_map):
    value_cols = [column_index_from_string(c) for c in final_map["value_columns"]]
    n = len(value_cols)

    # bottom-up (row method) — sum DATA rows by section, capex excluded
    comp_rev = [0.0] * n
    comp_exp = [0.0] * n
    rev_rows, exp_rows = [], []          # which DATA rows fed each section sum (evidence)
    for r in classified:
        if r["class"] != "DATA" or r.get("section") not in ("revenue", "expense"):
            continue
        (rev_rows if r["section"] == "revenue" else exp_rows).append(r["row"])
        for i, v in enumerate(_row_values(grid, r["row"], value_cols)):
            if isinstance(v, (int, float)):
                (comp_rev if r["section"] == "revenue" else comp_exp)[i] += v
    comp_rev = [round(x, 2) for x in comp_rev]
    comp_exp = [round(x, 2) for x in comp_exp]
    comp_noi = [round(comp_rev[i] - comp_exp[i], 2) for i in range(n)]

    # top-down (vendor-reported) — the grand-total rows
    def _num(vals):
        return [v if isinstance(v, (int, float)) else None for v in vals]
    act_rev = _num(_row_values(grid, final_map["revenue_end_row"], value_cols))
    act_exp = _num(_row_values(grid, final_map["expense_end_row"], value_cols))
    act_noi = _num(_row_values(grid, final_map["noi_row"], value_cols))

    def _match(a, c):
        if a is None or c is None:
            return None
        return abs(a - c) <= max(1.0, abs(a) * 0.005)

    months, any_noi_fail = [], False
    for i in range(n):
        noi_ok = _match(act_noi[i], comp_noi[i])
        if noi_ok is False:
            any_noi_fail = True
        months.append({
            "col": final_map["value_columns"][i],
            "rev": (act_rev[i], comp_rev[i], _match(act_rev[i], comp_rev[i])),
            "exp": (act_exp[i], comp_exp[i], _match(act_exp[i], comp_exp[i])),
            "noi": (act_noi[i], comp_noi[i], noi_ok),
        })
    noi_total_ok = months[-1]["noi"][2] if months else None
    return {"months": months, "overall_pass": (noi_total_ok is True) and (not any_noi_fail),
            "revenue_rows": rev_rows, "expense_rows": exp_rows,
            "vendor_rows": {"revenue_end": final_map["revenue_end_row"],
                            "expense_end": final_map["expense_end_row"], "noi": final_map["noi_row"]}}

## Module 13 — AI Failure Diagnosis

The NOI-**FAIL** branch of the diagram. When Module 12's NOI check fails, we hand the whole picture to a capable model (`gpt-4.1`) and ask it to name the root cause — no deterministic pre-triage, straight to the agent.

**Two possible verdicts:**
- **CLASSIFICATION_PROBLEM** — *we* mislabeled a row or boundary (a subtotal labeled DATA double-counts; a line item labeled a subtotal under-counts; a wrong section boundary). Fixable → the agent returns the exact row(s), the class we gave, and the class it should be.
- **DATA_PROBLEM** — the vendor's *own* totals are internally inconsistent (their stated NOI ≠ their Total Income − Total Expense), so no relabeling can fix it → the agent says which total is off and by how much.
- (**UNCERTAIN** if it genuinely can't tell.)

**Context sent** (everything needed for the decision): the detected structure (`final_map`), the exact per-column NOI discrepancy, and the **full raw grid annotated with our label on every row** — raw data and our interpretation side by side, so the model judges whose numbers are wrong. It's told the arithmetic figures rather than asked to re-sum (models are unreliable at arithmetic; the delta already points at the culprit row).

This is the **diagnosis only** — the auto-fix + re-run loop (apply the corrected class, re-verify, max 3 iterations) will be built on top of this once the diagnosis is trusted. It fires only on a real NOI failure; to exercise it we'll use small edited copies of real T12 files (change a cell to break the arithmetic) rather than injecting synthetic faults into the pipeline code.


In [34]:
# ── Module 13 — AI Failure Diagnosis (fires only when NOI verification FAILS) ─
# On a NOI failure we hand the WHOLE picture to a capable model and ask one thing:
# is the vendor's data internally inconsistent (DATA_PROBLEM — unfixable, tell the
# user), or did WE mislabel a row/boundary (CLASSIFICATION_PROBLEM — fixable)?
# Context = detected structure + the exact arithmetic discrepancy + the FULL raw
# grid annotated with our label on every row (raw data and our interpretation
# side by side, so the model can tell whose numbers are wrong).

from typing import Literal

_ROW_CLASS = Literal["METADATA", "SECTION_HEADER", "SUBSECTION_HEADER", "DATA",
                     "SUBSECTION_SUBTOTAL", "SECTION_SUBTOTAL", "COMPUTED_METRIC"]


class ProblemRow(BaseModel):
    row: int | None = Field(default=None, description="the sheet row number that is mislabeled")
    current_class: str | None = Field(default=None, description="the class our pipeline assigned")
    corrected_class: _ROW_CLASS | None = Field(default=None, description="the class it SHOULD be — EXACTLY one of the seven class names, nothing else appended")
    reason: str | None = Field(default=None, description="why, citing the row label and the arithmetic")


class DiagnosisResult(BaseModel):
    # every field defaulted / optional so a partial or null LLM response still parses
    verdict: Literal["DATA_PROBLEM", "CLASSIFICATION_PROBLEM", "UNCERTAIN"] = Field(default="UNCERTAIN", description="the root cause")
    problem_rows: list[ProblemRow] | None = Field(default=None, description="the mislabeled rows (for CLASSIFICATION_PROBLEM); empty otherwise")
    data_inconsistency: str | None = Field(default=None, description="which vendor total is internally inconsistent and by how much (for DATA_PROBLEM); else null")
    user_message: str = Field(default="", description="one or two plain-English sentences for the end user")
    reasoning: str = Field(default="", description="brief explanation of how the verdict was reached")


DIAGNOSIS_MODEL = "claude-opus-4-8"   # feedback-loop diagnosis (correctness-first)


def render_structure_summary(final_map):
    vc = final_map["value_columns"]
    code = (f"embedded ({final_map['code_pattern']})" if final_map["code_embedded_in_description"]
            else (f"column {final_map['code_column']}" if final_map["code_column"] else "none"))
    return (f"value_columns: {vc[0]}..{vc[-1]}   description_columns: {','.join(final_map['description_columns'])}   codes: {code}\n"
            f"month_header_row: {final_map['month_header_row']}   data_start_row: {final_map['data_start_row']}\n"
            f"revenue_end_row: {final_map['revenue_end_row']}   expense_end_row: {final_map['expense_end_row']}   noi_row: {final_map['noi_row']}\n"
            f"subsection_subtotal_rows (validated): {final_map.get('validated_subsection_subtotal_rows', [])}")


def render_discrepancy(noi_res):
    lines = [f"overall_pass = {noi_res['overall_pass']}",
             "Columns where NOI does NOT reconcile (vendor top-down vs our bottom-up):"]
    for m in noi_res["months"]:
        a, c, ok = m["noi"]
        if ok is False:
            ra, rc, _ = m["rev"]; ea, ec, _ = m["exp"]
            d = None if (a is None or c is None) else round(c - a, 2)
            lines.append(f"  col {m['col']}: NOI vendor={a} ours={c} (delta {d})   "
                         f"revenue vendor={ra} ours={rc}   expense vendor={ea} ours={ec}")
    return "\n".join(lines)


def render_grid_with_labels(grid, classified, final_map):
    """Full raw grid, each content row annotated with OUR class/section."""
    lab = {r["row"]: r for r in classified}
    L = [f"Sheet '{grid['sheet']}': rows 1-{grid['max_row']}, cols A-{get_column_letter(grid['max_col'])}.",
         "Each row: row | COL=value ...  =>  OUR: class/section   (text keeps leading spaces; * = bold)", ""]
    for grow in grid["rows"]:
        parts = []
        for c, cell in sorted(grow["cells"].items()):
            v = cell["value"]
            if v is None:
                continue
            star = "*" if cell["bold"] else ""
            v = round(v, 2) if isinstance(v, (int, float)) else repr(v)
            parts.append(f"{cell['letter']}{star}={v}")
        if not parts:
            continue
        r = lab.get(grow["row"])
        tag = f"   =>  OUR: {r['class']}/{r.get('section', '?')}" if r else ""
        L.append(f"{grow['row']:>4} | " + "  ".join(parts) + tag)
    return "\n".join(L)


DIAGNOSIS_PROMPT = """You are an expert auditor of commercial real estate T12 (trailing-12-month) income statements.

Our automated pipeline parsed this sheet, classified every row (METADATA / SECTION_HEADER / SUBSECTION_HEADER / DATA / SUBSECTION_SUBTOTAL / SECTION_SUBTOTAL / COMPUTED_METRIC), tagged each row's section (revenue / expense / capex), and then its NOI check FAILED: the bottom-up NOI (sum of the rows WE labeled as revenue/expense DATA) does not match the vendor's reported NOI.

Decide the ROOT CAUSE, exactly one of:
- CLASSIFICATION_PROBLEM — WE mislabeled one or more rows, so the bottom-up sum is wrong. Common cases: a subtotal row we labeled DATA (double-counts its children); a real line item we labeled a subtotal (under-counts); a wrong section boundary. The vendor's own totals are self-consistent. Return EVERY mislabeled row you can find — deep hierarchies usually have SEVERAL nested subtotal rows all wrongly labeled DATA (e.g. multiple rollups such as 'Total Rental Inc.', 'Total Income', or code-only totals ending in ...099/...999); list ALL of them in problem_rows, each with the class we gave and the correct class (EXACTLY one of the seven class names).
- DATA_PROBLEM — the vendor's OWN reported totals are internally inconsistent (e.g. their stated NOI is not their Total Income minus Total Expense), so NO relabeling can make it reconcile. Return which total is off and by how much.
- UNCERTAIN — the data does not let you decide.

KEY DECISION RULE — apply this first, using the per-section figures already given in the discrepancy:
- If our REVENUE total differs from the vendor's revenue total, OR our EXPENSE total differs from the vendor's expense total, then WE summed the wrong rows in that section -> CLASSIFICATION_PROBLEM. The mislabeled row is almost always in that section and its value equals the delta (a subtotal we labeled DATA over-counts; a line item we labeled a subtotal under-counts).
- If our revenue total MATCHES the vendor's revenue total AND our expense total MATCHES the vendor's expense total, then our NOI (revenue minus expense) is arithmetically correct by definition. Any remaining NOI mismatch then means the VENDOR's reported NOI does not equal their own revenue minus expense -> DATA_PROBLEM. Report the vendor's NOI, their revenue minus expense, and the difference. Do NOT invent a "missing row" in this case.

You are given our detected structure, the exact arithmetic discrepancy, and the FULL raw grid with OUR label on each row. Do NOT recompute long column sums yourself — reason from the row descriptions, our labels, and the discrepancy figures already given.

DETECTED STRUCTURE:
{structure}

NOI DISCREPANCY:
{discrepancy}

RAW GRID WITH OUR LABELS:
{grid_text}
"""


def run_diagnosis_agent(classified, grid, final_map, noi_res):
    """Robust: retries once, and on any parse failure returns an UNCERTAIN result
    instead of raising, so a malformed LLM response can't crash the pipeline."""
    llm = make_llm(provider="anthropic", model=DIAGNOSIS_MODEL, temperature=0)
    structured = llm.with_structured_output(DiagnosisResult)
    prompt = PromptTemplate(input_variables=["structure", "discrepancy", "grid_text"],
                            template=DIAGNOSIS_PROMPT)
    chain = prompt | structured
    inputs = {"structure":   render_structure_summary(final_map),
              "discrepancy": render_discrepancy(noi_res),
              "grid_text":   render_grid_with_labels(grid, classified, final_map)}
    last_err = None
    for _ in range(2):
        try:
            res = chain.invoke(inputs)
            res.problem_rows = res.problem_rows or []
            return res
        except Exception as e:
            last_err = e
    return DiagnosisResult(verdict="UNCERTAIN", problem_rows=[],
                           user_message="Diagnosis response could not be parsed; needs human review.",
                           reasoning=f"structured-output error: {str(last_err)[:150]}")

## Module 14 — Feedback loop (self-correction)

The NOI-FAIL branch, closed into a loop. When Module 12 fails, instead of just diagnosing once, we **iterate**: the diagnosis agent names the mislabeled rows → we apply its `corrected_class` to exactly those rows → re-run the NOI check → repeat (max 3). A fix is **accepted only if the money now reconciles** — the AI proposes *which* rows, the arithmetic decides *whether it's fixed*. This peels off nested-subtotal rows one diagnosis at a time, with no per-format code. Stops early on a `DATA_PROBLEM` (vendor issue). Outcomes: `FIXED` / `DATA_PROBLEM` / `UNCERTAIN` / `UNRESOLVED` (→ human review). The corrected labels and post-fix NOI overwrite `13_sections.json` / `14_noi.json`; the loop history is saved to `16_feedback.json`.


In [ ]:
# ── Module 14 — Feedback loop (diagnose → apply fix → re-verify, arithmetic-gated)
# When NOI fails, iterate: the diagnosis agent names the mislabeled rows, we apply
# its corrected classes, re-run the NOI check, and repeat (max 3). A fix is accepted
# ONLY if the money now reconciles — the AI proposes, the arithmetic decides. This
# self-corrects the nested-subtotal failures without any per-format code.

def apply_corrections(classified, problem_rows, locked=frozenset()):
    """Return a copy of `classified` with the diagnosis's corrected_class applied to
    the named rows (only rows/classes that are valid). Locked anchor rows are NEVER
    reclassified — the Stage-1 arithmetic already fixed them, so diagnosis can only
    move the rows between the anchors, never the NOI target itself."""
    fix = {pr["row"]: pr["corrected_class"] for pr in problem_rows
           if pr.get("row") is not None and pr.get("corrected_class") in CLASSES
           and pr["row"] not in locked}
    out = []
    for r in classified:
        r2 = dict(r)
        if r["row"] in fix:
            r2["class"] = fix[r["row"]]
        out.append(r2)
    return out


def run_feedback_loop(classified, grid, final_map, noi_res, max_iters=3):
    """Diagnose → apply → re-verify, up to max_iters. Accept a fix only when NOI
    reconciles. Returns dict(status, classified, noi, iterations, history, diagnosis).
    status ∈ {FIXED, DATA_PROBLEM, UNCERTAIN, UNRESOLVED}."""
    def _fails(noi):   # how many month columns still don't reconcile
        return sum(1 for m in noi["months"] if m["noi"][2] is False)
    locked = ({final_map.get("revenue_end_row"), final_map.get("expense_end_row"),
               final_map.get("noi_row")} | set(final_map.get("validated_subsection_subtotal_rows", []))) - {None}
    cur, cur_noi, history, diag = classified, noi_res, [], None
    for it in range(1, max_iters + 1):
        diag = run_diagnosis_agent(cur, grid, final_map, cur_noi)
        fixes = diag.problem_rows or []
        entry = {"iter": it, "verdict": diag.verdict, "proposed_fixes": len(fixes),
                 "noi_fail_cols_before": _fails(cur_noi),
                 "fixes": [{"row": pr.row, "from": pr.current_class, "to": pr.corrected_class,
                            "reason": pr.reason} for pr in fixes]}
        if diag.verdict == "DATA_PROBLEM":
            history.append(entry)
            return {"status": "DATA_PROBLEM", "classified": cur, "noi": cur_noi,
                    "iterations": it, "history": history, "diagnosis": diag}
        if diag.verdict != "CLASSIFICATION_PROBLEM" or not fixes:
            history.append(entry)
            return {"status": diag.verdict or "UNCERTAIN", "classified": cur, "noi": cur_noi,
                    "iterations": it, "history": history, "diagnosis": diag}
        cur = apply_corrections(cur, [pr.model_dump() for pr in fixes], locked)
        cur_noi = verify_noi(cur, grid, final_map)
        entry["noi_fail_cols_after"] = _fails(cur_noi)
        history.append(entry)
        if cur_noi["overall_pass"]:
            return {"status": "FIXED", "classified": cur, "noi": cur_noi,
                    "iterations": it, "history": history, "diagnosis": diag}
    return {"status": "UNRESOLVED", "classified": cur, "noi": cur_noi,
            "iterations": max_iters, "history": history, "diagnosis": diag}


## Orchestrator — the runner

Everything above only **defines** functions (like importing the src package). This cell is the only one that **executes**. `run_pipeline(INPUT_FILE)` walks the stages, **saves each stage's output** to `workflow_output/<file>/NN_stage.json`, and **loads the next stage's input from there** — the same file-contract the src pipeline uses. Each `### Module N` section: load its input → call the module → save its output → show it. Returns an `artifacts` dict for post-run inspection; the JSON files are the durable record.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ORCHESTRATOR — the runner. Every cell ABOVE only DEFINES functions; this is the
#  only cell that EXECUTES. It runs the pipeline stage by stage, SAVES each stage's
#  output to  workflow_output/<file>/  and LOADS the next stage's input from there
#  (the file contract between stages, exactly like the src pipeline).
#
#  Pick the file in the Setup cell (INPUT_FILE), run all cells top-to-bottom, then
#  this one. Inspect any stage's output as JSON, or via the returned `artifacts`.
# ═══════════════════════════════════════════════════════════════════════════════
# Logging verbosity. EXPLAIN keeps the bounded evidence on (arithmetic sums,
# subsection members, reconciliation rows, boundary keywords, fix reasons) — enough
# to explain every number. EXPLAIN_ALL adds the heavy dumps (per-row signal hit/miss
# for EVERY row + full row-number lists) — turn on only for a deep audit.
EXPLAIN     = True
EXPLAIN_ALL = True  # False


def run_pipeline(input_file):
    run_dir = OUTPUT_DIR / input_file.stem
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"input   : {input_file.name}")
    print(f"outputs : {run_dir}")

    ### ── Module 1 — File load ────────────────────────────────────────────────
    print("\n" + "=" * 78 + "\n### Module 1 — File load\n" + "=" * 78)
    wb, ws, info = load_file(input_file)
    save_json(run_dir / "01_info.json", info)
    for k, v in info.items():
        print(f"  {k:<13}: {v}")

    ### ── Module 2 — Pre-flight ───────────────────────────────────────────────
    print("\n" + "=" * 78 + "\n### Module 2 — Pre-flight\n" + "=" * 78)
    info = load_json(run_dir / "01_info.json")
    verdict, kind = preflight(input_file, info["sheet_names"])
    save_json(run_dir / "02_preflight.json", {"verdict": verdict, "kind": kind})
    print(f"  verdict: {verdict}   kind: {kind}")
    if verdict != "proceed":
        print(f"  ⚠ Rejected as '{kind}' — pipeline stops here.")
        return {"verdict": verdict, "kind": kind, "run_dir": run_dir}

    ### ── Module 3 — Raw extraction → cell_grid ───────────────────────────────
    print("\n" + "=" * 78 + "\n### Module 3 — Raw extraction → cell_grid\n" + "=" * 78)
    cell_grid = extract_cell_grid(input_file)
    save_json(run_dir / "03_cell_grid.json", cell_grid)
    cell_grid = load_cell_grid(run_dir / "03_cell_grid.json")          # load from disk
    print(f"  {len(cell_grid['rows'])} content rows | used range "
          f"{cell_grid['max_row']}x{cell_grid['max_col']} | "
          f"{len(cell_grid['merged_ranges'])} merged ranges")

    ### ── Module 4 — Format detection ─────────────────────────────────────────
    print("\n" + "=" * 78 + "\n### Module 4 — Format detection\n" + "=" * 78)
    result = detect_format(cell_grid)
    save_json(run_dir / "04_format.json", result)
    C, F = result["counts"], result["fractions"]
    print(f"  rows_with_text = {result['rows_with_text']}\n")
    print(f"    {'kind':<15} " + "  ".join(f"{c.upper():>11}" for c in COL_LETTERS))
    for k in KINDS:
        print(f"    {k:<15} " + "  ".join(f"{f'{C[k][c]}={F[k][c]}':>11}" for c in range(NCOLS)))
    print("\n  scores:")
    for k, v in result["scores"].items():
        print(f"    {k:<22}: {v}")
    print(f"\n  → format : {result['format']}  (confidence {result['confidence']}, threshold {DECISION_THRESHOLD})")
    print(f"    reason : {result['reason']}")
    fmt = result["format"]

    # ══════════════════════════════════════════════════════════════════════════
    #  KNOWN branch — one per-format extractor, then stop (known-format
    #  classification / NOI is handled by the existing src pipeline).
    # ══════════════════════════════════════════════════════════════════════════
    if fmt in ("code_in_a_desc_in_b", "embedded_code_in_a", "no_code"):
        print("\n" + "=" * 78 + f"\n### Modules 5-7 — Signal extraction ({fmt})\n" + "=" * 78)
        cell_grid = load_cell_grid(run_dir / "03_cell_grid.json")
        if fmt == "code_in_a_desc_in_b":
            value_cols = detect_value_columns(cell_grid, DESC_COL + 1)
            signal_rows = extract_row_signals(cell_grid, value_cols)
        elif fmt == "embedded_code_in_a":
            value_cols = detect_value_columns(cell_grid, EMB_DESC_COL + 1)
            signal_rows = extract_row_signals_embedded(cell_grid, value_cols)
        else:
            value_cols = detect_value_columns(cell_grid, NOCODE_DESC_COL + 1)
            signal_rows = extract_row_signals_no_code(cell_grid, value_cols)
        save_json(run_dir / "11_signals.json", signal_rows)
        vcl = [get_column_letter(c) for c in value_cols]
        print(f"  value cols {vcl[0]}..{vcl[-1]} ({len(value_cols)}) — {len(signal_rows)} rows\n")
        print_signal_rows(signal_rows)
        print("\n  (known-format classification / NOI is handled by the src pipeline.)")
        return {"format": fmt, "cell_grid": cell_grid, "result": result,
                "signal_rows": signal_rows, "run_dir": run_dir}

    if fmt != "unknown":
        print("\n  (no route for this format; stopping.)")
        return {"format": fmt, "result": result, "run_dir": run_dir}

    # ══════════════════════════════════════════════════════════════════════════
    #  UNKNOWN branch — 8a AI structure map → 8b validate → 9 extract →
    #  9b Stage-1 anchor lock (R−E=N decides) → 10 classify + fact override →
    #  11 section-assign (from locked anchors) → 12 NOI → 13/14 diagnose on fail
    # ══════════════════════════════════════════════════════════════════════════

    ### ── Module 8a — AI structure-map agent (reads the raw grid) ─────────────
    print("\n" + "=" * 78 + "\n### Module 8a — AI structure-map agent (raw grid -> structure map)\n" + "=" * 78)
    print(f"  invoking {AGENT_MODEL} (full structure map) ...\n")
    structure_map = run_structure_agent(cell_grid)                    # Pydantic model, from the raw grid
    save_json(run_dir / "09_structure_map.json", structure_map.model_dump())
    print(structure_map.model_dump_json(indent=2))

    ### ── Module 8b — Validate the AI map with arithmetic (no reconcile) ──────
    print("\n" + "=" * 78 + "\n### Module 8b — Validate the AI map (arithmetic; the map is trusted)\n" + "=" * 78)
    ai_map = load_json(run_dir / "09_structure_map.json")             # the AI map, trusted
    final_map, sub_results = build_final_map(ai_map, cell_grid)
    save_json(run_dir / "10_final_map.json", final_map)
    vc = final_map["value_columns"]
    print(f"  value_columns  : {vc[0]}..{vc[-1]} ({len(vc)})   month_header_row: {final_map['month_header_row']}   data_start_row: {final_map['data_start_row']}")
    print(f"  description_columns: {', '.join(final_map['description_columns'])}   code_embedded: {final_map['code_embedded_in_description']}   code_pattern: {final_map['code_pattern']}")
    print(f"  revenue_end/expense_end/noi: {final_map['revenue_end_row']} / {final_map['expense_end_row']} / {final_map['noi_row']}   (all from AI)")
    print(f"\n  subsection hierarchy - {len(sub_results)} sub-groups, {len(final_map['validated_subsection_subtotal_rows'])} arithmetically validated:")
    for r in sub_results:
        mark = "ok " if r["ok"] else "XX "
        tgt = "None" if r["target"] is None else f"{r['target']:,.2f}"
        mr = r["member_rows"]
        rng = f"{mr[0]}..{mr[-1]}" if len(mr) > 1 else (str(mr[0]) if mr else "-")
        print(f"    {mark} row {r['row']:>3} {r['name'][:24]:<24} = Σ {r['members']:>2} leaves (rows {rng:<10}) = {r['sum']:>15,.2f}  vs {tgt}")
        if EXPLAIN and mr:
            print(f"           leaf rows: {mr}")
    rev = _last_col_value(cell_grid, final_map["revenue_end_row"], vc[-1])
    exp = _last_col_value(cell_grid, final_map["expense_end_row"], vc[-1])
    noi = _last_col_value(cell_grid, final_map["noi_row"], vc[-1])
    if None not in (rev, exp, noi):
        diff = abs((rev - exp) - noi); ok = diff <= max(1.0, abs(noi) * 0.005)
        print(f"\n  arithmetic gate (col {vc[-1]}): {rev} - {exp} = {rev - exp} vs NOI {noi} -> {'PASS' if ok else 'FAIL'} (diff {diff:.2f})")

    ### ── Module 9 — Dynamic feature extraction ───────────────────────────────
    print("\n" + "=" * 78 + "\n### Module 9 — Dynamic signal extraction\n" + "=" * 78)
    final_map = load_json(run_dir / "10_final_map.json")
    signal_rows = extract_row_signals_dynamic(cell_grid, final_map)
    save_json(run_dir / "11_signals.json", signal_rows)
    dc = final_map["description_columns"]
    print(f"  value cols {vc[0]}..{vc[-1]} ({len(vc)}) | desc cols {', '.join(dc)} | "
          f"validated subsections: {len(final_map.get('validated_subsection_subtotal_rows', []))}\n")
    print_signal_rows(signal_rows)
    def _first(pred): return next((r for r in signal_rows if pred(r)), None)
    samples = [_first(lambda r: not r["has_values"]),
               _first(lambda r: r["has_values"] and not r["is_subsection_subtotal"]),
               _first(lambda r: r["is_subsection_subtotal"])]
    print("\n  feature-vector JSON — sample rows (incl. monthly values + signal_evidence):")
    for r in samples:
        if r:
            print(json.dumps({k: v for k, v in r.items() if not k.startswith("_")}, indent=2, default=str))
    print("\n  arithmetic sum-of-above evidence — every row whose value matched a running sum above it:")
    print_arithmetic_evidence(signal_rows, only_fired=True)

    ### ── Module 9b — Stage 1 anchor lock (keywords + AI nominate; R−E=N decides) ─
    print("\n" + "=" * 78 + "\n### Module 9b — Stage 1 anchor lock (arithmetic R − E = NOI decides)\n" + "=" * 78)
    lock = lock_anchors(signal_rows, final_map)
    a = lock["anchors"]
    if a:
        final_map["revenue_end_row"] = a["revenue_end_row"]
        final_map["expense_end_row"] = a["expense_end_row"]
        final_map["noi_row"]         = a["noi_row"]
        save_json(run_dir / "10_final_map.json", final_map)          # persist the LOCKED anchors
    save_json(run_dir / "10b_anchor_lock.json", {"status": lock["status"], "method": lock["method"],
              "anchors": a, "candidates": lock["candidates"], "passing": lock["passing"]})
    print(f"  candidates : {lock['candidates']}")
    print(f"  passing triples ({len(lock['passing'])}):")
    for (rr, eu, nr) in lock["passing"]:
        print(f"    R{rr} − E{eu} = N{nr}")
    tag = "" if lock["status"] == "AUTO" else "   ⚠ AMBIGUOUS → manual gate (raw file + these candidates)"
    print(f"  → status {lock['status']} ({lock['method']}) — locked "
          f"R={a['revenue_end_row'] if a else None} E={a['expense_end_row'] if a else None} N={a['noi_row'] if a else None}{tag}")

    ### ── Module 10 — Row classification (7-class, score-based) ───────────────
    print("\n" + "=" * 78 + "\n### Module 10 — Row classification (score-based, 7 classes)\n" + "=" * 78)
    signal_rows = load_json(run_dir / "11_signals.json")
    final_map = load_json(run_dir / "10_final_map.json")
    classified = classify_rows(signal_rows, final_map)
    classified = apply_anchor_overrides(classified, final_map)   # arithmetic-verified facts override the score
    save_json(run_dir / "12_classified.json", classified)
    print("  (fact override applied: locked anchors + validated subsection subtotals forced to their class)")
    print_classified_rows(classified)
    dist = Counter(r["class"] for r in classified)
    print("\n  class distribution:")
    for c in CLASSES:
        print(f"    {c:<20}: {dist.get(c, 0)}")
    label = ("every row" if EXPLAIN_ALL else "one example row per class")
    print("\n  " + "=" * 74 + f"\n  SCORE BREAKDOWN — {label} (winner vs runner-up, signals annotated):\n  " + "=" * 74)
    explain_scores(classified, final_map, which=("all" if EXPLAIN_ALL else "examples"))
    rec = reconcile_leaf_data(classified, cell_grid, final_map)
    print("\n  leaf-DATA reconciliation (sum of DATA-class rows vs the vendor grand total):")
    for side in ("revenue", "expense", "noi"):
        d = rec[side]
        n = f"  [{d['n']} DATA rows]" if "n" in d else ""
        print(f"    {side:<7}: {d['computed']:>15,.2f} vs {d['vendor']:>15,.2f}  {'PASS' if d['ok'] else 'FAIL'}{n}")
    if EXPLAIN_ALL or not rec["revenue"]["ok"]:
        print(f"      revenue DATA rows: {rec['revenue']['rows']}")
    if EXPLAIN_ALL or not rec["expense"]["ok"]:
        print(f"      expense DATA rows: {rec['expense']['rows']}")

    ### ── Module 11 — Section assignment (positional, from the locked anchors) ─
    print("\n" + "=" * 78 + "\n### Module 11 — Section assignment (from the Stage-1 locked anchors)\n" + "=" * 78)
    classified = load_json(run_dir / "12_classified.json")
    final_map = load_json(run_dir / "10_final_map.json")
    print(f"  locked anchors — revenue_end={final_map['revenue_end_row']}  "
          f"expense_end={final_map['expense_end_row']}  noi={final_map['noi_row']}")
    print("  (the keyword state machine is gone — a row's section is simply which region it falls between)")
    assign_sections(classified, final_map)
    save_json(run_dir / "13_sections.json", classified)
    sec = Counter(r["section"] for r in classified)
    print("  section assignment: " + "  ".join(f"{s}={sec.get(s, 0)}" for s in ("metadata", "revenue", "expense", "capex")))

    ### ── Module 12 — NOI verification (bottom-up vs top-down) ────────────────
    print("\n" + "=" * 78 + "\n### Module 12 — NOI verification (per month AND total)\n" + "=" * 78)
    classified = load_json(run_dir / "13_sections.json")
    final_map = load_json(run_dir / "10_final_map.json")
    res = verify_noi(classified, cell_grid, final_map)
    save_json(run_dir / "14_noi.json", res)
    hdr = f"  {'col':<4} {'revenue  act / comp':<26} {'expense  act / comp':<26} {'NOI  act / comp':<26}"
    print(hdr); print("  " + "-" * (len(hdr) - 2))
    def _fmt(t):
        a, c, ok = t
        mark = "  " if ok is None else ("ok" if ok else "XX")
        av = "  --  " if a is None else f"{a:,.0f}"
        cv = "  --  " if c is None else f"{c:,.0f}"
        return f"{av} / {cv} {mark}"
    for m in res["months"]:
        print(f"  {m['col']:<4} {_fmt(m['rev']):<26} {_fmt(m['exp']):<26} {_fmt(m['noi']):<26}")
    vr = res["vendor_rows"]
    print(f"\n  bottom-up computed from {len(res['revenue_rows'])} revenue + {len(res['expense_rows'])} expense DATA rows;"
          f"  vendor top-down rows: revenue={vr['revenue_end']} expense={vr['expense_end']} noi={vr['noi']}")
    if EXPLAIN_ALL or not res["overall_pass"]:
        print(f"    revenue DATA rows: {res['revenue_rows']}")
        print(f"    expense DATA rows: {res['expense_rows']}")
    print(f"\n  OVERALL NOI VERIFICATION: {'PASS' if res['overall_pass'] else 'FAIL'}")

    ### ── Modules 13-14 — Diagnosis + Feedback loop (only when NOI FAILS) ─────
    print("\n" + "=" * 78 + "\n### Modules 13-14 — Diagnosis + Feedback loop (only on NOI FAIL)\n" + "=" * 78)
    classified = load_json(run_dir / "13_sections.json")
    final_map = load_json(run_dir / "10_final_map.json")
    if res["overall_pass"]:
        print("  NOI PASSED — nothing to diagnose.")
        feedback = None
    else:
        print(f"  NOI FAILED — running diagnosis + feedback loop ({DIAGNOSIS_MODEL}) ...\n")
        feedback = run_feedback_loop(classified, cell_grid, final_map, res, max_iters=5)
        save_json(run_dir / "15_diagnosis.json", feedback["diagnosis"].model_dump())
        save_json(run_dir / "16_feedback.json", {"status": feedback["status"],
                  "iterations": feedback["iterations"], "history": feedback["history"]})
        save_json(run_dir / "13_sections.json", feedback["classified"])   # corrected labels
        save_json(run_dir / "14_noi.json", feedback["noi"])               # post-fix NOI
        classified, res = feedback["classified"], feedback["noi"]
        print(f"  loop status: {feedback['status']}  after {feedback['iterations']} iteration(s)")
        for h in feedback["history"]:
            after = h.get("noi_fail_cols_after")
            delta = f"  NOI-fail cols {h['noi_fail_cols_before']} → {after}" if after is not None else f"  NOI-fail cols {h['noi_fail_cols_before']}"
            print(f"    iter {h['iter']}: verdict={h['verdict']}  proposed_fixes={h['proposed_fixes']}{delta}")
            for fx in h.get("fixes", []):
                print(f"        row {fx['row']}: {fx['from']} → {fx['to']}")
                if fx.get("reason"):
                    print(f"           reason: {fx['reason']}")
        dg = feedback["diagnosis"]
        if dg.data_inconsistency:
            print(f"  data inconsistency: {dg.data_inconsistency}")
        if dg.user_message:
            print(f"  user message: {dg.user_message}")
        print(f"  final NOI: {'PASS' if res['overall_pass'] else 'FAIL'}")

    return {"format": fmt, "cell_grid": cell_grid, "result": result,
            "structure_map": structure_map, "final_map": final_map, "signal_rows": signal_rows,
            "classified": classified, "noi": res, "feedback": feedback, "run_dir": run_dir}


# Run the whole pipeline on the file chosen in the Setup cell.
artifacts = run_pipeline(INPUT_FILE)
